# 41 · Paper figures, assembled from saved results

This notebook draws every main and supplementary figure of the manuscript
(`docs/paper/draft.md`, section 7) from results that earlier notebooks saved. It fits no model and
runs no test.

**What it reads.**
- Saved tables under `results/`.
- A few arrays that upstream notebooks keep only in their stage cache: fitted curves, split-plot
  p-values, the pass-2 embedding, mouse depth and the Lake pseudobulk. Each is checked against a
  saved table before it is drawn.
- The private PT count matrices, for descriptive aggregates that no notebook saves (section 3):
  - segment pseudobulks and their log2 gradients;
  - specimen log2 CPM;
  - specimen means in 12 coordinate bins;
  - two reference gradients.

  Each is computed with the source notebook's own functions and must reproduce that notebook's
  saved statistics exactly before it is drawn.

Any quantity that still cannot be reproduced is drawn as a labelled placeholder and listed in
`missing_inputs.csv`.

**What it checks.** Before drawing, it asserts key counts against their tables and the internal
consistency of the revision tables (notebook 43). Every number printed on a figure is read from a
saved table or from an aggregate that has been checked against one. After drawing, a layout check
reports text that leaves the canvas or overlaps other text.

**Output.** `results/paper_figures/`: one PDF and one PNG per figure, 180 mm wide, plus
`figure_manifest.json` (panel sources) and `missing_inputs.csv`.

**Cohort caveat, shown beside every result.** Two control mice and two cortex sections from one
male human donor; all specimens male; the human section labelled medulla is healthy cortex. The
findings are descriptive.

| Figure | Content | Upstream notebooks |
|---|---|---|
| 1 | Design and the PT coordinate (R1) | 13, 35 |
| 2 | Specificity of pathway strategies (R2) | 31, 37 |
| 3 | Zonation against same-species ceilings (R3, revised claim) | 43, 37, 41 (means) |
| 4 | Robustness and external agreement (R4) | 37, 12 |
| 5 | Lead genes against direct segment measurements (R5) | 38, 40 |
| S1 | Coordinate limitations | 35, 36 |
| S2 | DPT specificity diagnostic | 36 (diagnostic) |
| S3 | Position-dependent pathways and programs | 37, 31 |
| S4 | Zonation amplitude | 39 |
| S5 | Lead-gene verification matrix | 38 |
| S6 | Opposite regional differences | 37, 31, 38 |
| S7 | Segment-label agreement with the cortex atlas | 38 |
| S8 | Superseded notebook 40 classes (kept for the record) | 40, 37 |

## 1 · Folders, style and input helpers

In [ ]:
import argparse
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import FancyBboxPatch, Patch, Rectangle

starts = ([Path(__file__).resolve().parent] if '__file__' in globals() else []) + [Path.cwd()]
project = next(p for s in starts for p in (s, *s.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
parser.add_argument('--revision-classes', help='folder under results/pt_revision_classes written by notebook 43')
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
output = results_root / 'paper_figures'
output.mkdir(parents=True, exist_ok=True)
REVISION_TAG = args.revision_classes or os.environ.get('PSEUDOSPACE_REVISION_CLASSES') or 'reviewed'
REVISION = results_root / 'pt_revision_classes' / REVISION_TAG

NOTEBOOK_LOGIC_VERSION = '41.paper_figures.2'
ALPHA = .05                      # discovery level used upstream (notebook 37)
MM = 1 / 25.4                    # inches per millimetre
WIDTH_MM = 180                   # journal double-column width
HUMAN, MOUSE, NULL, LIGHT, INK, MUTED, ACCENT = '#D55E00', '#0072B2', '#9A9A9A', '#D9D9D9', '#222222', '#666666', '#009E73'
CLASS_COLORS = {'conserved': '#009E73', 'reversal': '#CC79A7', 'mouse-only': MOUSE, 'human-only': HUMAN,
                'indeterminate': '#E3E3E3', 'neither': '#B8B8B8'}
SEGMENT_COLORS = {'S1': '#CDBCE3', 'S2': '#8A68BA', 'S3': '#3F2368'}   # ordinal, one hue, light to dark
CLASSES = ['conserved', 'reversal', 'mouse-only', 'human-only']
ENRICH_Q = .10   # notebook 40's enrichment threshold; asserted against its `enriched` column in Figure S8
SPECIMENS = {'mouse': ['Ctrl1A2', 'Ctrl1A4'], 'human': ['HUK1_COR1', 'HUK1_MED1']}
SHORT = {'Ctrl1A2': 'M1', 'Ctrl1A4': 'M2', 'HUK1_COR1': 'H1', 'HUK1_MED1': 'H2'}
SPECIES_OF = {s: sp for sp, names in SPECIMENS.items() for s in names}
SPECIMEN_MARKER = {'Ctrl1A2': 'o', 'Ctrl1A4': 's', 'HUK1_COR1': 'o', 'HUK1_MED1': 's'}

plt.rcParams.update({
    'font.family': ['Liberation Sans', 'DejaVu Sans'],   # Arial metrics; DejaVu supplies ★ ✓ ● ○
    'font.size': 6.5, 'axes.titlesize': 6.5, 'axes.titleweight': 'bold', 'axes.labelsize': 6.5,
    'xtick.labelsize': 6, 'ytick.labelsize': 6, 'legend.fontsize': 6, 'legend.frameon': False,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.linewidth': .6,
    'xtick.major.width': .6, 'ytick.major.width': .6, 'xtick.major.size': 2.5, 'ytick.major.size': 2.5,
    'xtick.major.pad': 1.5, 'ytick.major.pad': 1.5, 'axes.labelpad': 2, 'axes.titlepad': 3,
    'lines.linewidth': 1.0, 'mathtext.default': 'regular', 'pdf.fonttype': 42, 'svg.fonttype': 'none', 'savefig.dpi': 300})

SOURCES, MISSING, OUTPUTS, LAYOUT = {}, [], [], {}


def _record(figure, path, kind):
    path = Path(path)
    if path.is_relative_to(results_root):
        name = str(path.relative_to(results_root))
    else:   # private inputs are recorded relative to the data root, never as machine paths
        name = 'data/' + str(path.relative_to(data_root)) if path.is_relative_to(data_root) else path.name
    entry = {'file': name, 'kind': kind}
    if entry not in SOURCES.setdefault(figure, []):
        SOURCES[figure].append(entry)


def table(figure, relpath, **kwargs):
    """A required saved table: missing means the upstream notebook has not run, so stop."""
    path = results_root / relpath
    if not path.is_file():
        raise FileNotFoundError(f'{figure}: required input missing: {path}')
    _record(figure, path, 'table')
    return pd.read_csv(path, **kwargs)


def optional_table(figure, panel, relpath, quantity, upstream, **kwargs):
    """A table that an upstream notebook does not save yet; None (and a MISSING row) when absent."""
    path = results_root / relpath
    if path.is_file():
        _record(figure, path, 'table')
        return pd.read_csv(path, **kwargs)
    MISSING.append({'figure': figure, 'panel': panel, 'quantity': quantity, 'proposed_file': relpath,
                    'upstream': upstream})
    return None


def record_recomputed(figure, what):
    entry = {'file': f'notebook 41 section 3: {what}', 'kind': 'recomputed aggregate (checked against saved statistics)'}
    if entry not in SOURCES.setdefault(figure, []):
        SOURCES[figure].append(entry)


def note_missing(figure, panel, quantity, upstream, proposed_file=''):
    MISSING.append({'figure': figure, 'panel': panel, 'quantity': quantity, 'proposed_file': proposed_file,
                    'upstream': upstream})


def json_file(figure, relpath):
    path = results_root / relpath
    _record(figure, path, 'run manifest')
    return json.loads(path.read_text())


def cache_entry(figure, folder, stage, ext):
    """Newest stage-cache payload written by an upstream notebook (read-only; never recomputed here)."""
    root = results_root / folder / 'stage_cache'
    entries = []
    for sidecar in root.glob(f'{stage}__*.{ext}.key.json'):
        meta = json.loads(sidecar.read_text())
        payload = sidecar.with_name(sidecar.name[:-len('.key.json')])
        if meta.get('stage') == stage and payload.is_file():
            entries.append((meta['written_utc'], payload))
    if not entries:
        raise FileNotFoundError(f'{figure}: no stage-cache entry {stage} under {root}')
    path = sorted(entries)[-1][1]
    _record(figure, path, 'stage cache (validated against a saved table)')
    return path


def cached_arrays(figure, folder, stage):
    with np.load(cache_entry(figure, folder, stage, 'npz'), allow_pickle=False) as payload:
        return {k: payload[k] for k in payload.files}


def new_figure(height_mm):
    fig = plt.figure(figsize=(WIDTH_MM * MM, height_mm * MM))
    fig.height_mm = height_mm
    return fig


def axes_mm(fig, x, y, w, h, **kwargs):
    """Axes placed in millimetres from the bottom-left corner of the canvas."""
    return fig.add_axes([x / WIDTH_MM, y / fig.height_mm, w / WIDTH_MM, h / fig.height_mm], **kwargs)


def letter(ax, text, dx=-24, dy=3):
    ax.annotate(text, xy=(0, 1), xycoords='axes fraction', xytext=(dx, dy), textcoords='offset points',
                fontsize=9, fontweight='bold', ha='left', va='bottom', annotation_clip=False)


def title(ax, text, pad=3):
    ax.set_title(text, loc='left', pad=pad)


def placeholder(ax, message):
    ax.set_xticks([])
    ax.set_yticks([])
    for side in ax.spines.values():
        side.set_visible(True)
        side.set_color(LIGHT)
    ax.add_patch(Rectangle((0, 0), 1, 1, transform=ax.transAxes, facecolor='#F6F6F6', hatch='////',
                           edgecolor='#E2E2E2', lw=0))
    ax.text(.5, .5, message, transform=ax.transAxes, ha='center', va='center', fontsize=6, color=MUTED, wrap=True,
            bbox=dict(facecolor='white', edgecolor='none', pad=2))


def layout_report(fig, name):
    """Texts that leave the canvas or overlap other texts (a check, not a fix: inspect the PNG too).

    Measured at the PNG resolution, because hinted glyph widths change with dpi."""
    fig.set_dpi(300)
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    canvas = fig.bbox
    texts, tick_labels, drawn = [], set(), set()
    for ax in fig.axes:
        for axis in (ax.xaxis, ax.yaxis):
            for tick in axis.get_major_ticks() + axis.get_minor_ticks():
                tick_labels.update({id(tick.label1), id(tick.label2)})
            if ax.axison:
                for tick in axis._update_ticks():      # only the ticks inside the view limits are drawn
                    drawn.update({id(tick.label1), id(tick.label2)})
    for artist in fig.findobj(matplotlib.text.Text):
        if not artist.get_visible() or not artist.get_text().strip() or artist.figure is not fig:
            continue
        if id(artist) in tick_labels and id(artist) not in drawn:
            continue
        box = matplotlib.text.Text.get_window_extent(artist, renderer)   # the text only, not an annotation arrow
        if box.width > 0 and box.height > 0:
            texts.append((artist.get_text().replace('\n', ' ')[:40], box, artist.get_rotation() % 180))
    outside = [t for t, b, _ in texts if b.x0 < canvas.x0 - 3 or b.x1 > canvas.x1 + 3 or b.y0 < canvas.y0 - 3
               or b.y1 > canvas.y1 + 3]
    overlaps = []
    for i in range(len(texts)):
        for j in range(i + 1, len(texts)):
            if texts[i][2] == texts[j][2] and texts[i][2] not in (0, 90):
                continue   # parallel slanted labels: their upright boxes overlap even when the text does not
            a, b = texts[i][1], texts[j][1]
            ix = min(a.x1, b.x1) - max(a.x0, b.x0)
            iy = min(a.y1, b.y1) - max(a.y0, b.y0)
            if ix > 4.5 and iy > 4.5:
                overlaps.append((texts[i][0], texts[j][0]))
    LAYOUT[name] = {'texts': len(texts), 'outside_canvas': outside, 'overlapping_pairs': overlaps}
    return outside, overlaps


def show(path):
    """Preview a saved PNG when running inside Jupyter; a plain script run just writes the files."""
    try:
        from IPython import get_ipython
        from IPython.display import Image, display
    except ImportError:
        return
    if get_ipython() is not None:
        display(Image(filename=str(path), width=900))


def save(fig, name):
    outside, overlaps = layout_report(fig, name)
    for suffix in ('pdf', 'png'):
        fig.savefig(output / f'{name}.{suffix}', dpi=300)
    OUTPUTS.append(name)
    print(f'{name}: {fig.get_figwidth() / MM:.0f} × {fig.get_figheight() / MM:.0f} mm; '
          f'{len(outside)} text(s) outside the canvas, {len(overlaps)} overlapping text pair(s)')
    for item in outside:
        print('   outside:', item)
    for pair in overlaps:
        print('   overlap:', pair)
    plt.close(fig)
    show(output / f'{name}.png')


def place_labels(ax, xs, ys, labels, fontsize=5, step=7.5, style="italic"):
    """Greedy label placement: each label starts just above its point and moves up past earlier labels."""
    fig = ax.figure
    fig.canvas.draw()
    to_display = ax.transData.transform
    placed = []
    points = sorted(zip(xs, ys, labels), key=lambda t: (t[1], t[0]))
    for x, y, text in points:
        px, py = to_display((x, y)) * 72 / fig.dpi
        width = len(text) * fontsize * .55
        offset = 4.0
        while any(abs(px - qx) < (width + qw) / 2 + 2 and abs(py + offset - qy) < step for qx, qy, qw in placed):
            offset += step
        placed.append((px, py + offset, width))
        ax.annotate(text, (x, y), xytext=(0, offset), textcoords='offset points', ha='center', va='bottom',
                    fontsize=fontsize, style=style, arrowprops=dict(arrowstyle='-', lw=.3, color=MUTED, shrinkA=0, shrinkB=1))


SPECIMEN_BIN_COLUMNS = ['gene', 'specimen', 'bin_centre', 'mean']   # long format expected from upstream notebooks


def draw_specimen_bins(ax, bins, gene):
    """Each specimen's binned means for one gene, when the upstream notebook saved them."""
    if bins is None:
        return
    block = bins[bins.gene.eq(gene)]
    for name, rows in block.groupby('specimen'):
        ax.scatter(rows.bin_centre, rows['mean'], s=5, lw=0, marker=SPECIMEN_MARKER.get(name, 'o'), zorder=3, alpha=.8,
                   color=HUMAN if SPECIES_OF.get(name) == 'human' else MOUSE)


def read_pt_obs(figure):
    import anndata as ad
    path = results_root / 'minimal_pt_scfates' / 'cross_species_pt_scfates.h5ad'
    _record(figure, path, 'saved AnnData (obs only)')
    saved = ad.read_h5ad(path, backed='r')
    obs = saved.obs.copy()
    saved.file.close()
    return obs


def fmt_int(value):
    return f'{int(round(float(value))):,}'


print('Results root:', results_root)
print('Output folder:', output)

## 2 · Key counts asserted against their tables

These checks run before any drawing. Each count printed on a figure must agree with the table it
comes from, and with the rows from which that table was built.

In [ ]:
def check_key_counts():
    figure = 'checks'
    s2 = table(figure, 'pt_pathway_final/scfates/tables/table_S2_pathways.csv', index_col=0)
    funnel = table(figure, 'pt_pathway_final/scfates/tables/funnel.csv', index_col=0)['pathways']
    flags = pd.Series({'tested': len(s2), 'joint-test candidates': s2.candidate.eq(True).sum(),
                       'not specimen-sensitive': (s2.candidate.eq(True) & s2.specimen_sensitive.eq(False)).sum(),
                       'stable (confident)': s2.confident.eq(True).sum(),
                       'robust (registration + region)': s2.robust.eq(True).sum(),
                       'core (+ shared-gene, sex-unbiased)': s2.core.eq(True).sum()})
    assert flags.astype(int).to_dict() == funnel.astype(int).to_dict(), (flags, funnel)
    assert s2.loc[s2.core.eq(True)].robust.eq(True).all() and s2.loc[s2.robust.eq(True)].confident.eq(True).all()
    programs = table(figure, 'pt_pathway_final/scfates/tables/table_S4_programs.csv')
    assert programs.n_pathways.sum() == funnel['robust (registration + region)'] == s2.program.notna().sum()

    tests = table(figure, 'pt_pathway_final/scfates/tables/matched_and_joint_tests_all_partitions.csv')
    spec = table(figure, 'pt_pathway_final/scfates/tables/table_S1_strategy_specificity.csv', index_col=0)
    swaps = sorted(p for p in tests.partition.unique() if p.startswith('swap'))
    for stat, label in (('T_spatial', '8 · Gene GAM × smooth position (T_spatial)'), ('T_level', '4 · Gene GAM offset (T_level)')):
        block = tests[tests.statistic.eq(stat)]
        matched = block.effect.gt(0) & block.q_normal.le(ALPHA)
        joint = block.effect.gt(0) & block.q_joint.le(ALPHA)
        counts_m = matched.groupby(block.partition).sum()
        counts_j = joint.groupby(block.partition).sum()
        for name, counts in ((label, counts_m), (label + ' (joint test)', counts_j)):
            row = spec.loc[name]
            assert int(row.species_calls) == counts['species'], (name, counts)
            assert [int(row[f'relabel_calls_{s.split("+")[-1]}']) for s in swaps] == [counts[s] for s in swaps], name
    assert int(spec.loc['8 · Gene GAM × smooth position (T_spatial) (joint test)', 'species_calls']) == funnel['joint-test candidates']

    within = table(figure, 'pt_pathway_final/scfates/tables/within_species_controls.csv', index_col=0)
    block = tests[tests.statistic.isin(['T_level', 'T_spatial'])]
    recount = (block.assign(hit=block.effect.gt(0) & block.q_normal.le(ALPHA))
               .pivot_table(index='partition', columns='statistic', values='hit', aggfunc='sum'))
    assert recount.loc[['species', 'within_mouse', 'within_human']].to_numpy().tolist() == within.to_numpy().tolist()

    classes = table(figure, 'pt_zonation_classes/tables/gene_zonation_classes.csv', index_col=0)
    sizes = table(figure, 'pt_zonation_classes/tables/class_sizes_by_setting.csv', index_col=0)
    per_contrast = table(figure, 'pt_zonation_classes/tables/class_sizes_by_contrast.csv', index_col=0)
    r3 = table(figure, 'pt_zonation_classes/tables/table_R3_zonation_classes.csv', index_col=0)
    balanced = classes[classes.probe_balanced.eq(True)]
    order = list(sizes.columns)
    assert balanced['class'].value_counts().reindex(order).fillna(0).astype(int).tolist() == sizes.loc['primary'].tolist()
    for contrast in per_contrast.index:
        assert (balanced[contrast].value_counts().reindex(order).fillna(0).astype(int).tolist()
                == per_contrast.loc[contrast].tolist()), contrast
    assert r3['genes (primary, probe-balanced)'].reindex(order).astype(int).tolist() == sizes.loc['primary'].tolist()
    assert r3['S2 − S1 only'].reindex(order).astype(int).tolist() == per_contrast.loc['S2-S1'].tolist()
    by_class = table(figure, 'pt_zonation_classes/tables/robust_pathways_by_class.csv', index_col=0)
    robust_ids = set(s2.index[s2.robust.eq(True)])
    assert set(by_class.index) <= robust_ids
    absent = sorted(robust_ids - set(by_class.index))

    evidence = table(figure, 'pt_literature_deep/story_gene_evidence.csv', keep_default_na=False)
    verdicts = evidence.verdict.value_counts()
    claimed = evidence[~evidence.verdict.str.startswith('example')]

    pt = table(figure, 'pt_pathway_final/scfates/tables/segment_transitions_by_specimen.csv')
    assert sorted(pt.specimen.unique()) == sorted(SHORT)
    replication = table(figure, 'pt_pathway_final/scfates/tables/external_replication_summary.csv', index_col=0)
    replicated = (s2.loc[list(robust_ids)].external_q_within_robust.le(.10) & s2.loc[list(robust_ids)].external_z.gt(0)).sum()
    assert replicated == int(replication.loc['robust replicating (BH ≤ 0.10 in list)', 'Census cortex − male mouse (primary)'])
    print('Funnel:', funnel.astype(int).to_dict())
    print('Programs:', len(programs), '· robust pathways in the class table:', len(by_class), 'of', len(robust_ids),
          '· absent:', absent or 'none')
    print('Zonation classes (probe-balanced, primary):', sizes.loc['primary'].to_dict(), '· total', int(sizes.loc['primary'].sum()))
    print('Per contrast:', per_contrast.to_dict(orient='index'))
    print('Strategy calls (joint T_spatial): species 60-type check passed; within-species controls:', within.to_dict(orient='index'))
    print('Story verdicts over', len(claimed), 'claimed genes:', verdicts.to_dict())
    print('Robust pathways replicating against Census:', int(replicated))
    return {'robust_absent_from_class_table': absent, 'claimed_genes': len(claimed),
            'verdicts': verdicts.to_dict()}


KEY_COUNTS = check_key_counts()


def check_revision_tables():
    """Notebook 43's tables must agree with each other. Literal counts may change when labels change."""
    F = 'checks'
    rev = f'pt_revision_classes/{REVISION_TAG}/tables'
    classes = table(F, f'{rev}/gene_classes_symmetric.csv', index_col=0)
    counts = table(F, f'{rev}/class_counts_by_setting_and_contrast.csv')
    index = table(F, f'{rev}/conservation_index.csv')
    reframed = table(F, f'{rev}/robust_pathways_reframed.csv', index_col=0)
    decisions = table(F, f'{rev}/decisions.csv', index_col=0)['0']
    s2 = table(F, 'pt_pathway_final/scfates/tables/table_S2_pathways.csv', index_col=0)
    order = ['conserved', 'reversal', 'mouse-only', 'human-only', 'indeterminate', 'neither']
    balanced = classes[classes.probe_balanced.eq(True)]
    symmetric = counts[counts.setting.eq('symmetric (A3b + A8)')].set_index('contrast')
    for contrast in symmetric.index:
        assert (balanced[contrast].value_counts().reindex(order).fillna(0).astype(int).tolist()
                == symmetric.loc[contrast, order].astype(int).tolist()), contrast
    np.testing.assert_allclose(counts['mouse-only'] / counts['human-only'].clip(lower=1), counts['mouse-only ÷ human-only'], rtol=1e-12)
    np.testing.assert_allclose(index['index'], index.cross_corrected / np.sqrt(index.ceiling_human * index.ceiling_mouse), rtol=1e-9)
    assert ((index.ci_low <= index['index']) & (index['index'] <= index.ci_high)).all()
    primary = index[index.version.str.startswith('primary') & index.contrast.isin(['S2-S1', 'S3-early'])]
    wording = ('largely different genes (stands)' if (primary.ci_high < .5).all() else
               'largely conserved; apparent divergence is a cross-dataset ceiling' if (primary.ci_low >= .8).all() else 'partly conserved')
    assert decisions['D2 wording'] == wording, (decisions['D2 wording'], wording)
    ratio = symmetric['mouse-only ÷ human-only']
    asymmetry = ('asymmetry supported' if ratio['S2-S1'] >= 2 and ratio['S3c-early'] >= 2 else
                 'asymmetry confined to S2 − S1' if ratio['S2-S1'] >= 2 else 'no consistent asymmetry')
    assert decisions['D3 asymmetry'] == asymmetry, (decisions['D3 asymmetry'], asymmetry)

    def label(row):   # notebook 43's protocol rule, applied to the saved member counts
        zonated = row['n_mouse-led'] + row['n_human-led'] + row['n_shared'] + row['n_mixed']
        assert zonated == row.zonated_contributing
        if zonated < 3:
            return 'unresolved'
        for column, name in (('n_mouse-led', 'summary of mouse zonation'), ('n_shared', 'shared'), ('n_human-led', 'human-led')):
            if row[column] / zonated >= .5:
                return name
        return 'mixed'

    assert reframed.apply(label, axis=1).eq(reframed.category).all()
    member_columns = ['n_mouse-led', 'n_human-led', 'n_shared', 'n_mixed', 'n_not zonated', 'n_not classifiable']
    assert reframed[member_columns].sum(axis=1).eq(reframed.contributing_members).all()
    assert set(reframed.index) == set(s2.index[s2.robust.eq(True)])
    categories = reframed.category.value_counts()
    for key, labels in (('summary of mouse zonation', ['summary of mouse zonation']), ('shared', ['shared']),
                        ('human-led', ['human-led']), ('mixed or unresolved', ['mixed', 'unresolved'])):
        assert int(decisions[f'robust pathways: {key}']) == int(sum(categories.get(k, 0) for k in labels)), key
    summary = {'labels folder': REVISION_TAG, 'D2': decisions['D2 wording'], 'D3': decisions['D3 asymmetry'],
               'symmetric union': symmetric.loc['union S2-S1 + S3c-early', order].astype(int).to_dict(),
               'pathway labels': categories.to_dict()}
    print('Notebook 43 tables are internally consistent:', summary)
    return summary


KEY_COUNTS['revision'] = check_revision_tables()

## 3 · Descriptive aggregates recomputed from the private inputs

Some panels need per-gene or per-specimen means that no notebook saves. This section rebuilds
them from the PT count matrices, using the source notebooks' own functions:
- `rebuild_pt_expression` for the expression matrix;
- `group_sums` and `log_ratio` for segment pseudobulks and gradients (notebooks 39, 40 and 43);
- `paired_gradient` and `disattenuated` for the cross-species correlation (notebook 43);
- `amplitude_ratio` for the reference slopes (notebook 39);
- `gene_covariates` for detection (notebooks 37 and 38).

Only means are computed: no model is fitted and no test is run. Before anything is drawn, each
aggregate must reproduce saved statistics of its source notebook:

| Aggregate | Must reproduce |
|---|---|
| Gradients under notebook 43's labels | The human–mouse correlations (r, r*, gene counts) for S2 − S1, S3c − early and S3 − early; the noise-corrected amplitude; the set of genes classified for S2 − S1 |
| Gradients under the reviewed labels | Notebook 40's lead-gene S2 − S1 means; notebook 39's slopes on the male-microdissection and Lake references, and its naive slope |
| Coordinate bins | Notebook 37's grid and notebook 38's per-gene detection in each species |

Mouse cortical depth comes from the three upstream caches, which must agree. Notebook 43's labels
are read from its run manifest. They are the reviewed labels unless the manifest names a label
file, so the figure follows any relabelled rerun of notebook 43.

In [ ]:
FIGURE_S6_GENES = ['Slc22a6', 'Slc13a3', 'Cyp24a1']   # Slc22a7 dropped: its whole-PT difference is significant
N_COORDINATE_BINS = 12                                # notebooks 37 and 38: 13 edges over the common interval
CONTRASTS = {'S2-S1': ('S2', 'S1'), 'S3c-early': ('S3c', 'early'), 'S3-early': ('S3', 'early')}
SEGMENT_PARTS = {'S1': ['S1'], 'S2': ['S2'], 'S3': ['S3', 'S3deep'], 'S3c': ['S3'], 'early': ['S1', 'S2']}
MIN_COUNTS, REFERENCE_LEVEL, MIN_NUCLEI = 20, 3., 50   # notebooks 39, 40 and 43


def log2_cpm(counts_series, total):
    """Notebooks 39, 40 and 43: log2(CPM + 1) of summed counts."""
    return np.log2(counts_series / total * 1e6 + 1)


def revision_label_file():
    """The label file notebook 43 used for this folder (None for the reviewed labels)."""
    manifest = json_file('aggregates', f'pt_revision_classes/{REVISION_TAG}/run_manifest.json')
    if str(manifest['labels']).startswith('reviewed'):
        return None
    path = results_root / 'pt_revision_labels' / str(manifest['labels'])
    if not path.is_file():
        raise FileNotFoundError(f'Notebook 43 used labels {manifest["labels"]}, expected at {path}')
    _record('aggregates', path, 'segment labels (notebook 43 input)')
    return path


def recompute_aggregates():
    from scipy import sparse
    from pseudospace.pathway_inputs import rebuild_pt_expression
    from pseudospace.pathway_remodeling import gene_covariates
    from pseudospace.zonation_amplitude import group_sums, log_ratio, amplitude_ratio, ols_slope
    from pseudospace.zonation_reliability import paired_gradient, disattenuated

    F = 'aggregates'
    names = SPECIMENS['mouse'] + SPECIMENS['human']
    obs = read_pt_obs(F)
    obs = obs[obs.broad_tubule_marker_call.astype(str).eq('PT')].copy()
    orthologs = table(F, 'minimal_pt_scfates/ortholog_map_used.csv')
    paths = {n: data_root / 'tubule_by_gene' / f'{n}_tubule_by_gene_caleb.h5ad' for n in names}
    for path in paths.values():
        _record(F, path, 'private input (tubule-by-gene counts)')
    adata = rebuild_pt_expression(obs, paths, orthologs, target_sum=1e4)
    measured = adata.var.measured_in_both_inputs.to_numpy()
    gene_space = adata.var_names[measured]
    counts = sparse.csr_matrix(adata.layers['counts'], dtype=float)[:, measured]
    specimen = adata.obs['sample'].astype(str).to_numpy()
    reviewed = adata.obs.segment_class.astype(str).str.replace('PT-', '', regex=False).to_numpy()
    library = np.asarray(counts.sum(axis=1)).ravel()

    depths = [cached_arrays(F, folder, 'mouse_depth')['depth']
              for folder in ('pt_zonation_amplitude', 'pt_zonation_classes', f'pt_revision_classes/{REVISION_TAG}')]
    assert all(len(d) == len(specimen) and np.allclose(d, depths[0], equal_nan=True) for d in depths), 'cached depths disagree'
    depth = depths[0]

    def pseudobulk(segment):
        cortical = np.zeros(len(specimen), bool)
        for name in SPECIMENS['mouse']:
            limit = np.quantile(depth[(specimen == name) & (segment == 'S1')], .95)
            cortical |= (specimen == name) & (segment == 'S3') & (depth <= limit)
        region = np.where((segment == 'S3') & np.isin(specimen, SPECIMENS['mouse']) & ~cortical, 'S3deep', segment)
        keys, sums = group_sums(counts, np.char.add(np.char.add(specimen.astype(str), '|'), region.astype(str)))
        frame = pd.DataFrame(sums, index=keys, columns=gene_space)
        pb, lib = {}, {}
        for name in names:
            for label, parts in SEGMENT_PARTS.items():
                pb[(name, label)] = frame.loc[[f'{name}|{s}' for s in parts if f'{name}|{s}' in frame.index]].sum()
                lib[(name, label)] = float(library[(specimen == name) & np.isin(region, parts)].sum())
        pb, lib = pd.DataFrame(pb), pd.Series(lib)
        ours = pd.DataFrame({(c, n): log_ratio(pb[(n, late)], lib[(n, late)], pb[(n, early)], lib[(n, early)])
                             for c, (late, early) in CONTRASTS.items() for n in names}, index=gene_space)
        return pb, lib, ours

    def expressed(pb, specimens, parts):
        return set(gene_space[pb[[(n, p) for n in specimens for p in parts]].min(axis=1).ge(MIN_COUNTS).to_numpy()])

    def species_means(ours):
        return pd.DataFrame({f'{c} {sp} mean': ours[[(c, n) for n in SPECIMENS[sp]]].mean(axis=1)
                             for c in CONTRASTS for sp in ('mouse', 'human')})

    # Notebook 43's labels, and its saved statistics.
    label_file = revision_label_file()
    segment43 = reviewed
    if label_file is not None:
        supplied = pd.read_csv(label_file)
        mapped = (supplied.set_index(supplied.structure.astype(str)).segment.astype(str)
                  .str.replace('PT-', '', regex=False).reindex(adata.obs_names.astype(str)))
        segment43 = mapped.where(mapped.isin(['S1', 'S2', 'S3']), 'unlabelled').to_numpy()
    pb43, lib43, ours43 = pseudobulk(segment43)
    pairs = table(F, f'pt_revision_classes/{REVISION_TAG}/tables/pairwise_gradient_correlations.csv')
    for contrast, (late, early) in CONTRASTS.items():
        h = paired_gradient(ours43[[(contrast, n) for n in SPECIMENS['human']]].T.reset_index(drop=True))
        m = paired_gradient(ours43[[(contrast, n) for n in SPECIMENS['mouse']]].T.reset_index(drop=True))
        genes = pd.Index(sorted(expressed(pb43, SPECIMENS['human'], [late, early]) & expressed(pb43, SPECIMENS['mouse'], [late, early])))
        result = disattenuated(h, m, genes)
        saved = pairs[pairs.contrast.eq(contrast) & pairs.dataset_a.eq('human: ours') & pairs.dataset_b.eq('mouse: ours')].iloc[0]
        assert result['genes'] == saved.genes, (contrast, result['genes'], saved.genes)
        np.testing.assert_allclose([result['r'], result['r_corrected']], [saved.r, saved.r_corrected], rtol=1e-9)
    classes = table(F, f'pt_revision_classes/{REVISION_TAG}/tables/gene_classes_symmetric.csv', index_col=0)
    both = expressed(pb43, SPECIMENS['mouse'], ['S2', 'S1']) & expressed(pb43, SPECIMENS['human'], ['S2', 'S1'])
    assert set(classes.index[classes['S2-S1'].notna()]) == both, 'S2 − S1 classified genes differ from the count filter'
    manifest43 = json_file(F, f'pt_revision_classes/{REVISION_TAG}/run_manifest.json')
    reps = ours43.loc[sorted(both), [('S2-S1', n) for n in names]]
    signal = {sp: reps[[('S2-S1', n) for n in SPECIMENS[sp]]].to_numpy() for sp in SPECIMENS}
    signal = {sp: v.mean(axis=1).var(ddof=1) - v.var(axis=1, ddof=1).mean() / v.shape[1] for sp, v in signal.items()}
    np.testing.assert_allclose(np.sqrt(signal['human'] / signal['mouse']), manifest43['amplitude']['S2-S1'], rtol=1e-9)

    # Reviewed labels (notebooks 39 and 40), and their saved statistics.
    pb, lib, ours = (pb43, lib43, ours43) if label_file is None else pseudobulk(reviewed)
    means = species_means(ours)
    lead = table(F, 'pt_zonation_classes/tables/lead_gene_classes.csv', index_col=0)
    with_means = lead.index[lead['mouse S2−S1'].notna()]
    np.testing.assert_allclose(means.loc[with_means, 'S2-S1 mouse mean'], lead.loc[with_means, 'mouse S2−S1'].astype(float), atol=1e-12)
    np.testing.assert_allclose(means.loc[with_means, 'S2-S1 human mean'], lead.loc[with_means, 'human S2−S1'].astype(float), atol=1e-12)
    our_genes = pd.Index(sorted(expressed(pb, names, ['S2', 'S1'])))
    references = {}
    micro_path = data_root / 'external' / 'segment_microdissection' / 'GSE150338_Mouse_TPM_Replicates.txt.gz'
    _record(F, micro_path, 'private input (external microdissection)')
    micro = pd.read_csv(micro_path, sep='\t').drop_duplicates('Gene_symbol').set_index('Gene_symbol')
    import re
    per_mouse = {}
    for column in micro.columns:
        match = re.fullmatch(r'(m\d+)_PTS([123])_[a-z]', column)
        if match:
            per_mouse.setdefault((f'S{match.group(2)}', match.group(1)), []).append(column)
    micro = pd.DataFrame({key: np.log2(micro[cols].apply(pd.to_numeric, errors='coerce').clip(lower=0) + 1).mean(axis=1)
                          for key, cols in per_mouse.items()}).reindex(gene_space)
    segment_means = {s: micro[[c for c in micro.columns if c[0] == s]].mean(axis=1) for s in ('S1', 'S2')}
    keep = pd.concat([segment_means['S2'], segment_means['S1']], axis=1).max(axis=1).ge(REFERENCE_LEVEL)
    references['microdissection male (GSE150338)'] = (segment_means['S2'] - segment_means['S1'])[keep]
    lake = cached_arrays(F, 'pt_zonation_amplitude', 'lake_pseudobulk')
    meta = pd.DataFrame([g.split('|') for g in lake['groups'].astype(str)], columns=['donor', 'disease', 'sex', 'segment'])
    meta['nuclei'], meta['total'] = lake['nuclei'], lake['totals']
    sums = pd.DataFrame(lake['sums'], columns=gene_space)
    nuclei = meta.pivot_table(index='donor', columns='segment', values='nuclei', aggfunc='sum', fill_value=0)
    donors = nuclei.index[(nuclei.reindex(columns=['S1', 'S2', 'S3'], fill_value=0) >= MIN_NUCLEI).all(axis=1)]
    disease = meta.drop_duplicates('donor').set_index('donor').disease
    per_donor, levels = {}, {}
    for donor in [d for d in donors if disease[d] == 'normal']:
        def level(segment_name):
            rows = meta.index[meta.donor.eq(donor) & meta.segment.eq(segment_name)]
            return log2_cpm(sums.loc[rows].sum(), float(meta.loc[rows, 'total'].sum()))
        late_level, early_level = level('S2'), level('S1')
        per_donor[donor], levels[donor] = late_level - early_level, np.maximum(late_level, early_level)
    per_donor, levels = pd.DataFrame(per_donor).T, pd.DataFrame(levels).T
    references['Lake healthy human cortex'] = per_donor.mean()[levels.mean().ge(REFERENCE_LEVEL)]
    ratios = table(F, 'pt_zonation_amplitude/tables/amplitude_ratios.csv')
    for name, ref in references.items():
        genes = ref.index.intersection(our_genes)
        result = amplitude_ratio(means.loc[genes, 'S2-S1 human mean'], means.loc[genes, 'S2-S1 mouse mean'], ref.loc[genes])
        saved = ratios[ratios.contrast.eq('S2-S1') & ratios.reference.eq(name)].iloc[0]
        assert len(genes) == saved.genes, (name, len(genes), saved.genes)
        np.testing.assert_allclose([result['slope_human'], result['slope_mouse']], [saved.slope_human, saved.slope_mouse], rtol=1e-9)
    naive = ols_slope(means.loc[our_genes, 'S2-S1 mouse mean'], means.loc[our_genes, 'S2-S1 human mean'])
    np.testing.assert_allclose(naive, json_file(F, 'pt_zonation_amplitude/run_manifest.json')['naive_slope']['S2-S1'], rtol=1e-9)
    gradients = pd.DataFrame({'ours mouse': means['S2-S1 mouse mean'], 'ours human': means['S2-S1 human mean'],
                              **references}).loc[our_genes]

    # Specimen log2 CPM per segment (reviewed labels), for notebook 40's representative genes.
    segment_log2cpm = pd.DataFrame({f'{n} {s}': log2_cpm(pb[(n, s)], lib[(n, s)]) for n in names for s in ('S1', 'S2', 'S3c')})

    # Specimen means in coordinate bins, on notebooks 37 and 38's analysis set.
    support = adata.obs.groupby('sample', observed=True).shared_pseudospace.quantile([.01, .99]).unstack()
    coordinate = adata.obs.shared_pseudospace.to_numpy(float)
    inside = (coordinate >= support[.01].max()) & (coordinate <= support[.99].min())
    lognorm = sparse.csr_matrix(adata.layers['lognorm'], dtype=float)[inside]
    position, specimen_in = coordinate[inside], specimen[inside]
    lo = max(position[specimen_in == n].min() for n in names)
    hi = min(position[specimen_in == n].max() for n in names)
    np.testing.assert_allclose(np.linspace(lo, hi, 61), cached_arrays(F, 'pt_pathway_final/scfates', 'species_fit')['grid'], atol=1e-12)
    human = adata.obs.comparison_species.astype(str).eq('human').to_numpy()[inside]
    covariates = gene_covariates(lognorm, position, human, specimen_in, adata.var_names).set_index('gene')
    detection38 = table(F, 'pt_literature_deep/gene_level_lake_contrasts.csv', index_col=0)[['detection_mouse', 'detection_human']]
    np.testing.assert_allclose(covariates.loc[detection38.index, ['detection_mouse', 'detection_human']], detection38, atol=1e-12)
    edges = np.linspace(lo, hi, N_COORDINATE_BINS + 1)
    centres = (edges[:-1] + edges[1:]) / 2
    binned = np.clip(np.searchsorted(edges, position, side='right') - 1, 0, N_COORDINATE_BINS - 1)
    evidence = table(F, 'pt_literature_deep/story_gene_evidence.csv')
    programs = table(F, 'pt_pathway_final/scfates/tables/table_S4_programs.csv')
    wanted = sorted(set(evidence.gene) | {g for d in programs.drivers for g in d.split(', ')} | set(FIGURE_S6_GENES))
    var_index = adata.var_names.get_indexer(wanted)
    assert (var_index >= 0).all()
    rows = []
    for gene, column in zip(wanted, var_index):
        values = lognorm[:, column].toarray().ravel()
        for name in names:
            for b in range(N_COORDINATE_BINS):
                mask = (specimen_in == name) & (binned == b)
                if mask.any():
                    rows.append({'gene': gene, 'specimen': name, 'bin_centre': centres[b], 'mean': float(values[mask].mean())})
    bins = pd.DataFrame(rows, columns=SPECIMEN_BIN_COLUMNS)

    # Mouse snRNA (Census) segment means per sex for the story genes, as notebook 38 computed them.
    census_dir = data_root / 'external' / 'census_pt_segments'
    for name in ('mouse_pt_segment_pseudobulk_counts.csv.gz', 'mouse_pt_segment_pseudobulk_meta.csv'):
        _record(F, census_dir / name, 'private input (external Census snRNA)')
    sn_counts = pd.read_csv(census_dir / 'mouse_pt_segment_pseudobulk_counts.csv.gz', index_col=0)
    sn_meta = pd.read_csv(census_dir / 'mouse_pt_segment_pseudobulk_meta.csv', index_col=0)
    sn_meta = sn_meta[~sn_meta.development_stage.str.contains('embryo|E16|Theiler', case=False)].copy()
    sn_meta['segment'] = sn_meta.cell_type.map({'epithelial cell of proximal tubule segment 1': 'S1',
                                                'epithelial cell of proximal tubule segment 2': 'S2',
                                                'epithelial cell of proximal tubule segment 3': 'S3'})
    sn_meta = sn_meta[sn_meta.segment.notna() & sn_meta.n_cells.ge(MIN_NUCLEI)]
    complete = sn_meta.groupby('donor_id').segment.apply(lambda s: {'S1', 'S2', 'S3'}.issubset(set(s)))
    sn_meta = sn_meta[sn_meta.donor_id.isin(complete[complete].index)]
    story = list(evidence.gene)
    sn_log = np.log2(sn_counts.loc[sn_meta.index].div(sn_meta.total_counts_all_genes, axis=0).reindex(columns=story) * 1e6 + 1)
    sn_rows = []
    by_gene = evidence.set_index('gene')
    for sex in ('male', 'female'):
        block = sn_meta[sn_meta.sex.eq(sex)]
        contrast = pd.concat([sn_log.loc[rows.index[rows.segment.eq('S3')]].mean() - sn_log.loc[rows.index[rows.segment.eq('S1')]].mean()
                              for _, rows in block.groupby('donor_id')], axis=1).mean(axis=1)
        np.testing.assert_allclose(contrast.loc[story], by_gene.loc[story, f'mouse_sn_{sex}'], atol=1e-9)
        sn_means = pd.DataFrame({s: sn_log.loc[block.index[block.segment.eq(s)]].mean() for s in ('S1', 'S2', 'S3')})
        sn_rows += [{'gene': g, 'dataset': f'mouse snRNA, {sex}', **sn_means.loc[g].to_dict()} for g in story]
    sn_segment_means = pd.DataFrame(sn_rows)
    print(f'{len(adata):,} PT structures · {len(gene_space):,} genes · labels for notebook 43: '
          f'{label_file.name if label_file else "reviewed"} · {int(inside.sum()):,} structures on the coordinate interval')
    print('Reproduced: notebook 43 human–mouse r and r* (3 contrasts), its S2 − S1 gene set and amplitude; notebook 40 '
          'lead-gene means; notebook 39 slopes (2 references) and naive slope; notebook 37 grid; notebook 38 detection and '
          'mouse snRNA S3 − S1 contrasts.')
    return {'means43': species_means(ours43), 'means': means, 'gradients': gradients, 'segment_log2cpm': segment_log2cpm,
            'bins': bins, 'sn_segment_means': sn_segment_means}


AGG = recompute_aggregates()

## 4 · Figure 1 · Study design and a continuous cross-species PT coordinate (R1)

- **a** Design, with counts from notebook 13's structure tables.
- **b** The pass-2 Harmony embedding of the PT structures (dimensions 1 and 2 of the five used by
  the curve), coloured by reviewed segment. The black line traces the coordinate: the median
  embedding position in 30 equal-count bins of the saved scFates coordinate. scFates' own curve
  nodes were not saved. Right: specimen segment medians and transitions on the coordinate.
- **c** Coordinate agreement with external zonation (P1) and within segments (P1w).
- **d** Identifiability checks: absorption of an injected human-only gradient, sign agreement for
  externally species-specific genes, and the count-split arms.

In [ ]:
def read_embedding(figure, index):
    import h5py
    path = cache_entry(figure, 'minimal_pt_scfates', 'pass2_nephron_embedding', 'h5ad')
    with h5py.File(path, 'r') as handle:
        names = [n.decode() if isinstance(n, bytes) else str(n) for n in handle['obs']['_index'][:]]
        harmony = handle['obsm']['X_harmony'][:, :2]
    embedding = pd.DataFrame(harmony, index=names, columns=['H1', 'H2'])
    if not index.isin(embedding.index).all():
        raise ValueError('Cached pass-2 embedding does not contain every saved PT structure.')
    return embedding.loc[index]


def figure_1():
    F = 'Figure 1'
    obs = read_pt_obs(F)
    obs['segment'] = obs.segment_class.astype(str).str.replace('PT-', '', regex=False)
    counts = obs.groupby(['sample', 'segment'], observed=True).size().unstack()
    filters = table(F, 'minimal_pt_scfates/pass2_nephron_filter_counts.csv')
    medians = table(F, 'minimal_pt_scfates/pt_segment_order_by_specimen.csv', index_col=0)
    recomputed = obs.groupby(['sample', 'segment_class'], observed=True).shared_pseudospace.median().unstack()
    np.testing.assert_allclose(recomputed.loc[medians.index, ['PT-S1', 'PT-S2', 'PT-S3']], medians[['PT-S1', 'PT-S2', 'PT-S3']], rtol=1e-6)
    transitions = table(F, 'pt_pathway_final/scfates/tables/segment_transitions_by_specimen.csv')
    p1 = table(F, 'pt_reconstruction_v2/nb35/p1_external_concordance.csv')
    p1w = table(F, 'pt_reconstruction_v2/nb35/p1w_within_segment_concordance.csv')
    arms = table(F, 'pt_reconstruction_v2/nb36/arm_metrics.csv', index_col=0)
    absorption = table(F, 'pt_reconstruction_v2/nb35/p2_absorption.csv', index_col=0)
    positive = table(F, 'pt_reconstruction_v2/nb35/positive_control_recovery.csv')
    split = table(F, 'pt_reconstruction_v2/nb35/g6_count_split.csv')
    embedding = read_embedding(F, obs.index)
    total = int(filters.n_structures.sum())
    retained = int(filters.loc[filters.pass2_decision.eq('retained'), 'n_structures'].sum())
    assert retained == int(filters.loc[filters.coarse_class.ne('Glomerulus') & filters.pass2_decision.eq('retained'), 'n_structures'].sum())
    assert int(filters.loc[filters.coarse_class.eq('PT'), 'n_structures'].sum()) == len(obs)

    fig = new_figure(162)
    # a · design
    ax = axes_mm(fig, 0, 139, 180, 20)
    ax.set_xlim(0, 180)
    ax.set_ylim(0, 20)
    ax.axis('off')
    seg_total = counts.sum()
    boxes = [
        'Visium HD, 2-µm bins\nM1, M2: control mouse\nH1, H2: human cortex,\none donor',
        'Panoptic segmentation\nof tubule cross-sections\non H&E',
        f'Tubule-by-gene\nprofiles\n{total:,} structures',
        f'Reviewed clustering,\ntwo integration passes\n{retained:,} nephron\nstructures',
        f'PT subset\n{len(obs):,} structures\nS1 {seg_total["S1"]:,} · S2 {seg_total["S2"]:,}\nS3 {seg_total["S3"]:,}',
        'Nonbranching scFates\ncurve on the integrated\nembedding: PT\ncoordinate, 0 to 1']
    box_w, gap = 26.4, 4.0
    for k, text in enumerate(boxes):
        x0 = 1 + k * (box_w + gap)
        ax.add_patch(FancyBboxPatch((x0, 1), box_w, 17, boxstyle='round,pad=0,rounding_size=1.5', facecolor='#F4F4F4',
                                    edgecolor='#BDBDBD', lw=.6))
        ax.text(x0 + box_w / 2, 9.5, text, ha='center', va='center', fontsize=6, linespacing=1.25)
        if k < len(boxes) - 1:
            ax.annotate('', xy=(x0 + box_w + gap - .4, 9.5), xytext=(x0 + box_w + .4, 9.5),
                        arrowprops=dict(arrowstyle='-|>', color=MUTED, lw=.7, mutation_scale=6))
    letter(ax, 'a', dx=0, dy=-1)

    # b · embedding with the coordinate trace, and specimen medians/transitions
    order = obs.shared_pseudospace.rank(method='first')
    bins = pd.qcut(order, 30, labels=False)
    trace = embedding.groupby(bins.to_numpy()).median()
    lim_x = np.quantile(embedding.H1, [.002, .998])
    lim_y = np.quantile(embedding.H2, [.002, .998])
    for k, species in enumerate(('mouse', 'human')):
        ax = axes_mm(fig, 12 + k * 46, 82, 40, 44)
        mask = obs.comparison_species.astype(str).eq(species).to_numpy()
        for seg in ('S1', 'S2', 'S3'):
            m = mask & obs.segment.eq(seg).to_numpy()
            ax.scatter(embedding.H1[m], embedding.H2[m], s=.6, lw=0, color=SEGMENT_COLORS[seg], alpha=.6, rasterized=True)
        ax.plot(trace.H1, trace.H2, color=INK, lw=1.1)
        ax.scatter(trace.H1.iloc[[0]], trace.H2.iloc[[0]], s=10, color=INK, zorder=3)
        ax.set(xlim=lim_x, ylim=lim_y, xticks=[], yticks=[], xlabel='Harmony 1', ylabel='Harmony 2' if k == 0 else '')
        names = SPECIMENS[species]
        title(ax, f'{species.capitalize()} ({", ".join(SHORT[n] for n in names)}; n = {int(mask.sum()):,})')
        if k == 0:
            letter(ax, 'b', dx=-14)
            handles = [Line2D([], [], marker='o', ls='', color=SEGMENT_COLORS[s], ms=3.5, label=s) for s in ('S1', 'S2', 'S3')]
            handles.append(Line2D([], [], color=INK, lw=1.1, marker='o', ms=2.5, label='coordinate trace\n(dot = start)'))
            ax.legend(handles=handles, loc='lower left', fontsize=5.5, handletextpad=.3, borderaxespad=.2, labelspacing=.3)
    ax = axes_mm(fig, 118, 82, 58, 44)
    names = SPECIMENS['mouse'] + SPECIMENS['human']
    table_t = transitions.pivot(index='specimen', columns='transition', values='position')
    for yi, name in enumerate(names[::-1]):
        color = HUMAN if SPECIES_OF[name] == 'human' else MOUSE
        ax.plot([0, 1], [yi, yi], color=LIGHT, lw=.8, zorder=1)
        for seg in ('S1', 'S2', 'S3'):
            ax.scatter(medians.loc[name, f'PT-{seg}'], yi, s=26, color=SEGMENT_COLORS[seg], edgecolor=color, lw=.8, zorder=3)
        for transition in table_t.columns:
            ax.plot([table_t.loc[name, transition]] * 2, [yi - .28, yi + .28], color=color, lw=1.3, zorder=2)
    ax.set_yticks(range(len(names)), [f'{SHORT[n]} ({SPECIES_OF[n]})' for n in names[::-1]])
    ax.set(xlim=(0, 1), ylim=(-1.5, len(names) - .4), xlabel='PT coordinate (scFates)')
    for transition in table_t.columns:
        ax.text(table_t[transition].mean(), len(names) - .45, transition, ha='center', va='bottom', fontsize=5.5, color=MUTED)
    handles = [Line2D([], [], marker='o', ls='', mfc=SEGMENT_COLORS['S2'], mec=INK, ms=3.5, label='segment median'),
               Line2D([], [], marker='|', ls='', color=INK, ms=5, mew=1.2, label='reviewed transition')]
    ax.legend(handles=handles, loc='lower center', fontsize=5.5, ncol=2, handletextpad=.2, borderaxespad=.1)
    title(ax, 'Segment medians and transitions per specimen', pad=9)

    # c · P1 and P1w
    ax = axes_mm(fig, 27, 14, 36, 50)
    refs = [('mouse_sn_male', 'mouse', 'Mouse snRNA,\nmale donors'), ('mouse_micro', 'mouse', 'Mouse micro-\ndissection'),
            ('human_sn_halfB', 'human', 'Human cortex\nsnRNA (half B)')]
    for yi, (ref, species, label) in enumerate(refs):
        color = HUMAN if species == 'human' else MOUSE
        rows = p1[p1.reference.eq(ref) & p1.species.eq(species)].set_index('coordinate').rho
        for offset, (coordinate, filled) in zip((-.18, .18), (('SCF13 fold-protected', True), ('segment labels (notebook 34 style)', False))):
            value = rows[coordinate]
            ax.barh(yi + offset, value, height=.32, color=color if filled else 'white', edgecolor=color, lw=.8)
            ax.text(value + .01, yi + offset, f'{value:.2f}', va='center', fontsize=5.5)
    ax.set_yticks(range(len(refs)), [r[2] for r in refs])
    ax.invert_yaxis()
    ax.set(xlim=(0, 1.0), xlabel='Spearman ρ across genes')
    handles = [Patch(facecolor=INK, label='coordinate,\nfold-protected'), Patch(facecolor='white', edgecolor=INK, label='segment labels')]
    ax.legend(handles=handles, loc='lower right', fontsize=5.5, borderaxespad=0, handlelength=1)
    title(ax, 'P1 · vs external S3 − early')
    letter(ax, 'c', dx=-52)
    ax = axes_mm(fig, 76, 14, 40, 50)
    primary = p1w[p1w.primary]
    groups = [('mouse', 'S1'), ('mouse', 'S2'), ('mouse', 'S3'), ('human', 'early'), ('human', 'S3')]
    for gi, (species, seg) in enumerate(groups):
        color = HUMAN if species == 'human' else MOUSE
        for offset, (coordinate, bar_color) in zip((-.19, .19), (('SCF13 fold-protected', color), ('read depth (nuisance reference)', LIGHT))):
            cell = primary[primary.coordinate.eq(coordinate) & primary.species.eq(species) & primary.segment.eq(seg)].rho
            ax.bar(gi + offset, cell.mean(), width=.36, color=bar_color)
            ax.scatter(np.full(len(cell), gi + offset), cell, s=4, color=INK, zorder=3, lw=0)
    for species in ('mouse', 'human'):
        means = [primary[primary.coordinate.eq('SCF13 fold-protected') & primary.species.eq(species) & primary.segment.eq(seg)].rho.mean()
                 for sp, seg in groups if sp == species]
        np.testing.assert_allclose(np.mean(means), arms.loc['SCF13', f'P1w {species}'], rtol=1e-6)
    ax.axhline(0, color=INK, lw=.5)
    ax.set_xticks(range(len(groups)), [f'{sp}\n{seg}' for sp, seg in groups])
    ax.set_ylabel('Spearman ρ across genes')
    ax.set_ylim(-.25, .42)
    handles = [Patch(facecolor=MOUSE, label='coordinate (mouse)'), Patch(facecolor=HUMAN, label='coordinate (human)'),
               Patch(facecolor=LIGHT, label='read-depth order'), Line2D([], [], marker='o', ls='', color=INK, ms=2, label='specimen')]
    ax.legend(handles=handles, loc='upper right', fontsize=5.5, ncol=1, handlelength=1, borderaxespad=0)
    title(ax, f'P1w · within segments\n(mean: mouse {arms.loc["SCF13", "P1w mouse"]:.2f}, human {arms.loc["SCF13", "P1w human"]:.2f})', pad=3)

    # d · identifiability
    ax = axes_mm(fig, 141, 56, 36, 8)
    for yi, coordinate in enumerate(['SCF13', 'DPT13']):
        value = absorption.loc[coordinate, 'median_ratio']
        ax.scatter(value, yi, s=12, color=INK if coordinate == 'SCF13' else NULL, zorder=3)
        ax.text(value - .06, yi, f'{value:.2f}', va='center', ha='right', fontsize=5.5)
    ax.axvline(1, color=MUTED, lw=.6, ls='--')
    ax.set(xlim=(0, 1.25), ylim=(-.7, 1.7), xticks=[0, .5, 1], xlabel='absorption ratio')
    ax.set_yticks([0, 1], ['scFates', 'DPT'])
    ax.invert_yaxis()
    title(ax, 'Injected human-only gradient\n(1 = kept, 0 = absorbed)')
    letter(ax, 'd', dx=-30, dy=12)
    ax = axes_mm(fig, 141, 31, 36, 11)
    sets = [('opposite sign', 'opposite'), ('mouse-zonated, human-flat', 'mouse only'), ('all positive controls', 'all')]
    for yi, (label, short) in enumerate(sets):
        rows = positive[positive.set.eq(label)].set_index('coordinate')
        ax.scatter(rows.loc['SCF13 fold-protected', 'sign agreement'], yi, s=12, color=INK, zorder=3)
        ax.scatter(rows.loc['segment labels', 'sign agreement'], yi, s=12, facecolor='white', edgecolor=INK, zorder=3, lw=.7)
    ax.set_yticks(range(len(sets)), [f'{s[1]} ({int(positive[positive.set.eq(s[0])].genes.iloc[0])})' for s in sets])
    ax.set(xlim=(.84, 1.0), ylim=(-.6, 2.6), xticks=[.85, .9, .95, 1], xlabel='sign agreement')
    ax.invert_yaxis()
    title(ax, 'Species-specific genes (n)\n● coordinate  ○ segment labels')
    ax = axes_mm(fig, 141, 6, 36, 12)
    arm_labels = {'a': 'A', 'b': 'saved', 'c': 'B'}
    xs = np.arange(len(split))
    robust_col = next(c for c in split if c.startswith('robust called'))
    core_col = next(c for c in split if c.startswith('core called'))
    ax.bar(xs - .19, split[robust_col], .36, color=INK, label='robust/66')
    ax.bar(xs + .19, split[core_col], .36, color=NULL, label='core/42')
    for x, (r, c) in enumerate(split[[robust_col, core_col]].to_numpy()):
        ax.text(x - .19, r + 1, str(int(r)), ha='center', fontsize=5.5)
        ax.text(x + .19, c + 1, str(int(c)), ha='center', fontsize=5.5)
    ax.set_xticks(xs, [arm_labels.get(a.split(' ')[0], a) for a in split.arm])
    ax.set(xlim=(-.6, 2.6), ylim=(0, 104), yticks=[0, 30, 60], ylabel='called')
    ax.legend(loc='upper right', fontsize=5.5, ncol=2, handlelength=.8, columnspacing=.8, borderaxespad=0)
    title(ax, 'Count split, A builds, B tests\n(earlier 66 robust, 42 core)', pad=3)
    fig.text(.005, .003, 'Two control mice (M1, M2) and two cortex sections (H1, H2) from one male human donor; H2 is labelled '
             'medulla but is cortex. Descriptive.', fontsize=5.5, color=MUTED)
    save(fig, 'figure_1')


figure_1()

## 5 · Figure 2 · Only position-dependent pathway screens are specific (R2)

- **a** The three balanced contrasts of four specimens and the split-plot error terms.
- **b** Pathways called (BH q ≤ 0.05) for the species split (filled) and both relabelings (open),
  per strategy, with NCDR, null rate and decoy-calibrated calls from notebook 37's Table S2.
- **c** Decoy-estimated FDP as the threshold is lowered. The curves are the target–decoy
  summary (`pathway_decoys.decoy_fdp`) of the saved per-pathway scores. Each curve is checked
  against the minimum FDP in the saved strategy table.
- **d** Within-species controls. **e** Mean decoy z² against pathway size (sextiles), with the
  saved inflation fits; DESeq2 gene counts from notebook 31's saved table.

Specific strategies are drawn in black and the others in grey. Orange and blue are kept for the
two species.

In [ ]:
from matplotlib.transforms import blended_transform_factory
from pseudospace.pathway_decoys import decoy_fdp

STRATEGY_LABELS = {
    '1 · Whole-PT pseudobulk + signed GSEA': '1  Whole-PT pseudobulk, GSEA',
    '2 · S1/S2/S3 pseudobulk + signed GSEA': '2  S1/S2/S3 pseudobulk, GSEA',
    '3a · Pathway-score GAM, offset': '3a Pathway score, offset',
    '3b · Pathway-score GAM, shape': '3b Pathway score, shape',
    '4 · Gene GAM offset (T_level)': '4  Gene offset (T_level)',
    '5 · Gene GAM total (T_total)': '5  Gene total (T_total)',
    '6 · Gene GAM × S1/S2/S3 steps': '6  Gene × S1/S2/S3 steps',
    '7 · Gene GAM × 6 equal bins': '7  Gene × 6 equal bins',
    '8 · Gene GAM × smooth position (T_spatial)': '8  Gene × position (T_spatial)',
    '8 · Gene GAM × smooth position (T_spatial) (joint test)': '8  T_spatial, joint test',
    'Gene GAM: smooth position beyond S1/S2/S3 steps': '    Position beyond steps',
    '8b · T_spatial, size-only random sets': '8b T_spatial, size-only null',
    '8c · T_spatial, preranked GSEA': '8c T_spatial, preranked GSEA'}


def decoy_curves(figure, spec):
    """Target–decoy FDP curves from saved per-pathway scores, each checked against the saved minimum FDP."""
    tests = table(figure, 'pt_pathway_final/scfates/tables/matched_and_joint_tests_all_partitions.csv')
    gsea = table(figure, 'pt_pathway_method_selection/conventional_signed_gsea_all_partitions.csv')
    swaps = sorted(p for p in tests.partition.unique() if p.startswith('swap'))
    frames = {}
    whole = gsea[gsea.scope.eq('whole_PT')].assign(score=lambda d: d.NES.abs())
    frames['1 · Whole-PT pseudobulk + signed GSEA'] = whole
    frames['2 · S1/S2/S3 pseudobulk + signed GSEA'] = (gsea[gsea.family.eq('segments')].assign(score=lambda d: d.NES.abs())
                                                       .groupby(['partition', 'pathway_id']).score.max().reset_index())
    for stat, name in (('T_level', '4 · Gene GAM offset (T_level)'), ('T_discrete_spatial', '6 · Gene GAM × S1/S2/S3 steps'),
                       ('T_spatial', '8 · Gene GAM × smooth position (T_spatial)')):
        rows = tests[tests.statistic.eq(stat) & ~tests.partition.str.startswith('within')]
        frames[name] = rows.assign(score=rows.z_matched)
        frames[name + ' (joint test)'] = rows.assign(score=rows.z_joint)
    curves = {}
    for name, frame in frames.items():
        scores = frame.pivot_table(index='pathway_id', columns='partition', values='score', aggfunc='first')
        scores = scores[['species', *[s for s in scores.columns if s.startswith('swap')]]].dropna()
        curve = decoy_fdp(scores['species'], scores.drop(columns='species'))
        np.testing.assert_allclose(curve.loc[curve.target_calls >= 10, 'fdp'].min(), spec.loc[name, 'min_fdp_10plus'], atol=1e-9)
        curves[name] = curve
    return curves, tests, swaps


def figure_2():
    F = 'Figure 2'
    spec = table(F, 'pt_pathway_final/scfates/tables/table_S1_strategy_specificity.csv', index_col=0)
    within = table(F, 'pt_pathway_final/scfates/tables/within_species_controls.csv', index_col=0)
    inflation = table(F, 'pt_pathway_final/scfates/tables/decoy_inflation_by_statistic.csv', index_col=0)
    deseq = table(F, 'pt_pathway_method_selection/conventional_deseq2_all_partitions.csv',
                  usecols=['partition', 'scope', 'padj'])
    curves, tests, swaps = decoy_curves(F, spec)
    de_counts = deseq[deseq.scope.eq('whole_PT')].assign(hit=lambda d: d.padj.le(ALPHA)).groupby('partition').hit.sum()
    relabel_columns = [c for c in spec.columns if c.startswith('relabel_calls_')]
    fig = new_figure(150)

    # a · contrasts
    ax = axes_mm(fig, 16, 108, 34, 28)
    signs = np.array([[-1, -1, 1, 1], [-1, 1, 1, -1], [-1, 1, -1, 1]], float) / 2
    ax.imshow(signs, cmap=matplotlib.colors.ListedColormap(['#E8E8E8', '#9E9E9E']), vmin=-.5, vmax=.5, aspect='auto')
    for (i, j), v in np.ndenumerate(signs):
        ax.text(j, i, '+½' if v > 0 else '−½', ha='center', va='center', fontsize=6.5)
    ax.set_xticks(range(4), ['M1', 'M2', 'H1', 'H2'])
    ax.set_yticks(range(3), ['species', 'relabeling 1', 'relabeling 2'])
    ax.tick_params(length=0)
    for side in ax.spines.values():
        side.set_visible(False)
    title(ax, 'Three balanced contrasts')
    letter(ax, 'a', dx=-42)
    fig.text(4 / WIDTH_MM, 100 / fig.height_mm,
             'Split-plot error terms\n'
             '• species offset (T_level): specimens\n   within species, 2 df\n'
             '• species × position (T_spatial):\n   specimen × position, 2(k − 1) df\n'
             '• each relabeling is one draw of the\n   section-level null\n'
             '• a term shared by H1 and H2 (donor,\n   panel, region) enters the species\n   contrast and cancels in relabelings',
             fontsize=5.8, va='top', linespacing=1.3)

    # b · strategies
    show = spec.drop(index=[i for i in spec.index if i.endswith('(joint test)') and 'T_spatial)' not in i])
    ax = axes_mm(fig, 92, 84, 44, 60)
    yy = np.arange(len(show))[::-1]
    for y_value, (name, row) in zip(yy, show.iterrows()):
        relabel_values = [row[c] for c in relabel_columns]
        ax.plot([min(relabel_values + [row.species_calls]), max(relabel_values + [row.species_calls])], [y_value] * 2,
                color=LIGHT, lw=1, zorder=1)
        for value, marker in zip(relabel_values, ('s', 'D')):
            ax.scatter(value, y_value, marker=marker, s=11, facecolor='white', edgecolor=NULL, lw=.7, zorder=2)
        ax.scatter(row.species_calls, y_value, s=16, color=INK if row.specific_nb31_rule else NULL, zorder=3)
    ax.set_yticks(yy, [STRATEGY_LABELS.get(n, n) for n in show.index], fontsize=5.8)
    ax.set_xscale('symlog', linthresh=10)
    ax.set_xlim(-.5, 2500)
    ax.set_xticks([0, 10, 100, 1000], ['0', '10', '100', '1,000'])
    ax.set_xlabel('Pathways called (BH q ≤ 0.05)')
    title(ax, 'Species split versus relabelings')
    letter(ax, 'b', dx=-92)
    columns = transform = blended_transform_factory(fig.transFigure, ax.transData)
    for x_mm, header in ((147, 'NCDR'), (159, 'null'), (171, 'decoy-\ncalibrated')):
        ax.text(x_mm / WIDTH_MM, len(show) - .3, header, transform=columns, ha='right', va='bottom', fontsize=5.8, fontweight='bold')
    for y_value, (name, row) in zip(yy, show.iterrows()):
        ncdr = f'{row.ncdr:.2f}' if np.isfinite(row.ncdr) else '–'
        for x_mm, text in ((147, ncdr), (159, f'{row.null_rate:.0%}'), (171, fmt_int(row.decoy_calls_fdp10))):
            ax.text(x_mm / WIDTH_MM, y_value, text, transform=transform, ha='right', va='center', fontsize=5.8,
                    color=INK if row.specific_nb31_rule else MUTED)
    handles = [Line2D([], [], marker='o', ls='', color=INK, ms=3.5, label='species, specific'),
               Line2D([], [], marker='o', ls='', color=NULL, ms=3.5, label='species, not specific'),
               Line2D([], [], marker='s', ls='', mfc='white', mec=NULL, ms=3, label='relabeling 1'),
               Line2D([], [], marker='D', ls='', mfc='white', mec=NULL, ms=3, label='relabeling 2')]
    ax.legend(handles=handles, loc='upper left', bbox_to_anchor=(-.02, -.13), ncol=2, fontsize=5.5, handletextpad=.2,
              columnspacing=.8)

    # c · decoy FDP curves
    ax = axes_mm(fig, 13, 24, 44, 36)
    styles = (('1 · Whole-PT pseudobulk + signed GSEA', 'whole-PT GSEA', NULL, ':'),
              ('2 · S1/S2/S3 pseudobulk + signed GSEA', 'S1/S2/S3 GSEA', NULL, '--'),
              ('4 · Gene GAM offset (T_level)', 'T_level', NULL, '-'),
              ('6 · Gene GAM × S1/S2/S3 steps', 'S1/S2/S3 steps', '#5A5A5A', '-.'),
              ('8 · Gene GAM × smooth position (T_spatial)', 'T_spatial, matched', '#5A5A5A', '-'),
              ('8 · Gene GAM × smooth position (T_spatial) (joint test)', 'T_spatial, joint', INK, '-'))
    for name, label, color, style in styles:
        curve = curves[name]
        curve = curve[curve.target_calls >= 5]
        ax.plot(curve.target_calls, curve.fdp, color=color, ls=style, lw=1.0 if 'joint' not in label else 1.4, label=label)
    ax.axhline(.10, color=MUTED, lw=.5)
    ax.text(1600, .11, 'FDP 10%', ha='right', va='bottom', fontsize=5.5, color=MUTED)
    ax.set(xscale='log', xlabel='Species pathways called (threshold lowered →)', ylabel='Decoy-estimated FDP', ylim=(0, 1.05))
    ax.legend(loc='upper left', bbox_to_anchor=(-.12, -.22), ncol=3, fontsize=5.5, handlelength=2.0, columnspacing=.6)
    title(ax, 'Target–decoy false-discovery estimate')
    letter(ax, 'c', dx=-26)

    # d · within-species controls
    ax = axes_mm(fig, 74, 24, 38, 36)
    labels = [i.replace('\n', ' ') for i in within.index]
    x = np.arange(len(within))
    ax.bar(x - .19, within.T_level, .36, color=LIGHT, label='offset (T_level)')
    ax.bar(x + .19, within.T_spatial, .36, color=INK, label='smooth position (T_spatial)')
    for xi, (a_value, b_value) in enumerate(within[['T_level', 'T_spatial']].to_numpy()):
        ax.text(xi - .19, a_value + 2, str(int(a_value)), ha='center', va='bottom', fontsize=5.5)
        ax.text(xi + .19, b_value + 2, str(int(b_value)), ha='center', va='bottom', fontsize=5.5)
    ax.set_xticks(x, ['species\nsplit', 'mouse vs\nmouse', 'human vs\nhuman section'])
    ax.set(ylabel='Pathways called (matched test)', ylim=(0, max(within.max()) * 1.25))
    ax.legend(loc='upper right', fontsize=5.5, borderaxespad=0)
    title(ax, 'Specimen differences are offsets')
    letter(ax, 'd', dx=-26)

    # e · decoy inflation
    ax = axes_mm(fig, 130, 24, 46, 36)
    species_sizes = tests[tests.statistic.eq('T_spatial') & tests.partition.eq('species')].set_index('pathway_id').n_genes
    for stat, color in (('T_level', NULL), ('T_spatial', INK)):
        z = tests[tests.statistic.eq(stat) & tests.partition.isin(swaps)].pivot(index='pathway_id', columns='partition', values='z_matched')
        sizes = species_sizes.reindex(z.index)
        binned = (z ** 2).mean(axis=1).groupby(pd.qcut(sizes, 6), observed=True).mean()
        ax.plot([interval.mid for interval in binned.index], binned.to_numpy(), 'o', color=color, ms=3)
        grid = np.linspace(10, 300, 50)
        a, rho = inflation.loc[stat, 'a'], inflation.loc[stat, 'rho']
        ax.plot(grid, a * (1 + (grid - 1) * rho), color=color, lw=1, label=f'{stat}: a = {a:.2f}, ρ = {rho:.3f}')
    ax.axhline(1, color=MUTED, lw=.5)
    ax.set(xlabel='Pathway size (genes)', ylabel='Mean decoy z² (1 = calibrated)', xlim=(0, 310), xticks=[0, 100, 200, 300])
    ax.legend(loc='upper left', fontsize=5.5, borderaxespad=0)
    relabel = [fmt_int(de_counts[s]) for s in sorted(de_counts.index) if s.startswith('swap')]
    ax.text(.03, .74, f'DESeq2 genes, whole PT (padj ≤ 0.05):\nspecies {fmt_int(de_counts["species"])}; relabelings {", ".join(relabel)}',
            transform=ax.transAxes, ha='left', va='top', fontsize=5.5)
    title(ax, 'Specimen noise inflates offset screens')
    letter(ax, 'e', dx=-26)
    save(fig, 'figure_2')


figure_2()

## 6 · Figure 3 · Beyond a conserved core, PT zonation is largely species-specific against same-species ceilings (R3)

All inputs come from notebook 43 (`results/pt_revision_classes/<labels>/`), apart from the species
means in panel a. The human segment labels may still change: workstream 1 is validating them, and
notebook 43 will rerun into its folder. The figure therefore asserts that notebook 43's tables
agree with each other (section 2) rather than any literal count, and prints every number from the
tables.

- **a** Per-gene S1 → S2 zonation: mean of the two mice against mean of the two human sections
  (log2 pseudobulk ratio), for probe-balanced genes classified in S2 − S1. Points are coloured by
  the symmetric class. The means come from section 3 under notebook 43's labels. The class counts
  are those of `class_counts_by_setting_and_contrast.csv`.
- **b** Left: noise-corrected correlations of gene gradients between datasets of the same species
  (the ceilings) and between species, rat excluded. Right: the conservation index with its 95%
  interval, the cross-species r* and both same-species ceilings, for each contrast and for our
  data and external data. Dashed lines mark the pre-registered decision thresholds 0.5 and 0.8.
- **c** Mouse-only and human-only gene counts for every setting and contrast. The colour is
  log2(mouse-only ÷ human-only); bold rows are the revision's primary (symmetric) setting.
- **d** The robust position-dependent pathways: their contributing members by category, grouped
  by the pathway label, and the label totals.

In [ ]:
MEMBER_COLORS = {'mouse-led': MOUSE, 'shared': CLASS_COLORS['conserved'], 'human-led': HUMAN, 'mixed': '#7F7F7F',
                 'not zonated': '#E3E3E3', 'not classifiable': '#F4F4F4'}
LABEL_ORDER = ['summary of mouse zonation', 'shared', 'human-led', 'mixed', 'unresolved']
SETTING_SHORT = {'v1 (notebook 40 primary)': 'v1 (notebook 40)', 'v1 + A6 female snRNA': 'v1 + female snRNA',
                 'v1 + A7 female microdissection': 'v1 + female micro.', 'A3b amplitude-matched': 'amplitude-matched',
                 'A8 external flat confirmation': 'external flat check', 'symmetric (A3b + A8)': 'symmetric (primary)',
                 'symmetric + A6 female snRNA': 'symmetric + female snRNA',
                 'symmetric + A7 female microdissection': 'symmetric + female micro.'}
COMPARISON_COLORS = {'within human': HUMAN, 'within mouse': MOUSE, 'human–mouse': INK}


def figure_3():
    F = 'Figure 3'
    rev = f'pt_revision_classes/{REVISION_TAG}/tables'
    classes = table(F, f'{rev}/gene_classes_symmetric.csv', index_col=0)
    counts = table(F, f'{rev}/class_counts_by_setting_and_contrast.csv')
    index = table(F, f'{rev}/conservation_index.csv')
    pairs = table(F, f'{rev}/pairwise_gradient_correlations.csv')
    reframed = table(F, f'{rev}/robust_pathways_reframed.csv', index_col=0)
    decisions = table(F, f'{rev}/decisions.csv', index_col=0)['0']
    means = AGG['means43']
    record_recomputed(F, "species-mean S2 − S1 gradients under notebook 43's labels")
    fig = new_figure(226)

    # a · per-gene S2 − S1 by symmetric class
    ax = axes_mm(fig, 14, 168, 50, 48)
    balanced = classes[classes.probe_balanced.eq(True) & classes['S2-S1'].notna()]
    x, y = means.loc[balanced.index, 'S2-S1 mouse mean'], means.loc[balanced.index, 'S2-S1 human mean']
    row = counts[counts.setting.eq('symmetric (A3b + A8)') & counts.contrast.eq('S2-S1')].iloc[0]
    for name in ['neither', 'indeterminate', *CLASSES]:
        mask = balanced['S2-S1'].eq(name).to_numpy()
        small = name in ('neither', 'indeterminate')
        ax.scatter(x[mask], y[mask], s=1.5 if small else 5, lw=0, color=CLASS_COLORS[name], alpha=.6 if small else .95,
                   rasterized=True, label=f'{name} ({int(row[name]):,})', zorder=1 if small else 3)
    ax.set_xlim(np.quantile(x, [.0005, .9995]) * 1.05)
    ax.set_ylim(np.quantile(y, [.0005, .9995]) * 1.05)
    ax.axhline(0, color=NULL, lw=.5)
    ax.axvline(0, color=NULL, lw=.5)
    ax.set(xlabel='Mouse S2 − S1 (log2, mean of M1, M2)', ylabel='Human S2 − S1 (log2, mean of H1, H2)')
    ax.legend(loc='upper left', fontsize=5.3, markerscale=1.8, handletextpad=.1, borderaxespad=0, labelspacing=.25,
              title=f'symmetric class ({len(balanced):,} genes)', title_fontsize=5.3)
    title(ax, 'Per-gene S1 → S2 zonation')
    letter(ax, 'a', dx=-26)

    # b, left · ceilings and cross-species correlations across all dataset pairs (rat excluded)
    ax = axes_mm(fig, 78, 168, 30, 48)
    shown = pairs[pairs.comparison.isin(list(COMPARISON_COLORS)) & pairs.r_corrected.notna()
                  & pairs.contrast.isin(['S2-S1', 'S3-early'])]
    for k, contrast in enumerate(['S2-S1', 'S3-early']):
        for j, comparison in enumerate(COMPARISON_COLORS):
            values = shown.loc[shown.contrast.eq(contrast) & shown.comparison.eq(comparison), 'r_corrected']
            xs = k * 4 + j + np.random.default_rng(10 * k + j).uniform(-.18, .18, len(values))
            ax.scatter(xs, values.clip(-.1, 1.05), s=7, lw=0, color=COMPARISON_COLORS[comparison],
                       label=comparison if k == 0 else None)
            if len(values):
                ax.plot([k * 4 + j - .3, k * 4 + j + .3], [values.median()] * 2, color=COMPARISON_COLORS[comparison], lw=1)
    ax.set_xticks([1, 5], ['S2 − S1', 'S3 − early'])
    ax.set(ylim=(-.1, 1.05), xlim=(-.7, 6.7), ylabel='noise-corrected r* of gene gradients')
    ax.axhline(0, color=NULL, lw=.4)
    ax.legend(loc='center', fontsize=5.2, handletextpad=.1, borderaxespad=0, bbox_to_anchor=(.5, .44))
    title(ax, 'Dataset pairs (bar: median)')
    letter(ax, 'b', dx=-26)

    # b, right · conservation index against both ceilings
    ax = axes_mm(fig, 140, 168, 36, 48)
    rows = index[~index.version.str.contains('alternative')].copy()
    rows['label'] = [f'{c.replace("-", " − ")} · {"ours" if v.startswith("primary") else "external"}'
                     for c, v in zip(rows.contrast, rows.version)]
    yy = np.arange(len(rows))[::-1]
    for yi, (_, r) in zip(yy, rows.iterrows()):
        ax.plot([r.ci_low, r.ci_high], [yi, yi], color=INK, lw=1.1)
        ax.scatter(r['index'], yi, s=14, color=INK, zorder=3)
        ax.scatter(r.cross_corrected, yi, s=10, facecolor='white', edgecolor=INK, lw=.7, zorder=3)
        ax.scatter(r.ceiling_human, yi, s=12, marker='^', color=HUMAN, zorder=3)
        ax.scatter(r.ceiling_mouse, yi, s=12, marker='^', color=MOUSE, zorder=3)
    for value in (.5, .8):
        ax.axvline(value, color=MUTED, ls='--', lw=.6)
    ax.set_yticks(yy, rows.label, fontsize=5.5)
    ax.set(xlim=(0, 1), xlabel='index, r* and ceilings')
    handles = [Line2D([], [], marker='o', ls='-', color=INK, ms=3, label='conservation index (95% CI)'),
               Line2D([], [], marker='o', ls='', mfc='white', mec=INK, ms=3, label='cross-species r*'),
               Line2D([], [], marker='^', ls='', color=HUMAN, ms=3, label='human ceiling'),
               Line2D([], [], marker='^', ls='', color=MOUSE, ms=3, label='mouse ceiling')]
    ax.legend(handles=handles, loc='upper left', bbox_to_anchor=(-.75, -.17), ncol=2, fontsize=5.2, handletextpad=.1,
              columnspacing=.6)
    title(ax, 'Conservation index vs ceilings')

    # c · species-only counts across settings and contrasts
    ax = axes_mm(fig, 50, 112, 92, 34)
    settings = list(dict.fromkeys(counts.setting))
    contrasts = ['S2-S1', 'S3c-early', 'S3-early', 'union S2-S1 + S3c-early']
    grid = counts.set_index(['setting', 'contrast'])
    ratio = np.array([[np.log2(max(grid.loc[(s, c), 'mouse-only'], .5) / max(grid.loc[(s, c), 'human-only'], .5))
                       for c in contrasts] for s in settings])
    cmap = matplotlib.colors.LinearSegmentedColormap.from_list('species', [HUMAN, '#F2F2F2', MOUSE])
    image = ax.imshow(np.clip(ratio, -4, 4), cmap=cmap, vmin=-4, vmax=4, aspect='auto')
    for i, s in enumerate(settings):
        for j, c in enumerate(contrasts):
            m, h = int(grid.loc[(s, c), 'mouse-only']), int(grid.loc[(s, c), 'human-only'])
            ax.text(j, i, f'{m} / {h}', ha='center', va='center', fontsize=5.5, color='white' if abs(ratio[i, j]) > 2.6 else INK)
    ax.set_xticks(range(len(contrasts)), ['S2 − S1', 'S3c − early', 'S3 − early', 'union'])
    ax.set_yticks(range(len(settings)), [SETTING_SHORT.get(s, s) for s in settings], fontsize=5.6)
    for tick, s in zip(ax.get_yticklabels(), settings):
        if s.startswith('symmetric (A3b'):
            tick.set_fontweight('bold')
    ax.xaxis.tick_top()
    ax.tick_params(length=0)
    for side in ax.spines.values():
        side.set_visible(False)
    cax = axes_mm(fig, 146, 116, 2, 26)
    bar = fig.colorbar(image, cax=cax)
    bar.set_ticks([-4, -2, 0, 2, 4], labels=['≤ −4', '−2', '0', '2', '≥ 4'])
    bar.ax.tick_params(labelsize=5.3)
    bar.set_label('log2(mouse-only ÷ human-only)', fontsize=5.5)
    ax.set_title(f'Species-only genes, mouse / human (probe-balanced): {decisions["D3 asymmetry"]}', loc='left', pad=12)
    letter(ax, 'c', dx=-118, dy=12)

    # d · robust pathways reframed
    ax = axes_mm(fig, 52, 14, 62, 84)
    member_columns = [f'n_{k}' for k in MEMBER_COLORS]
    frame = reframed.assign(order=reframed.category.map({k: i for i, k in enumerate(LABEL_ORDER)}),
                            mouse_share=reframed['n_mouse-led'] / reframed.contributing_members.clip(lower=1))
    frame = frame.sort_values(['order', 'mouse_share'], ascending=[True, False])
    yy = np.arange(len(frame))
    left = np.zeros(len(frame))
    for column in member_columns:
        share = frame[column] / frame.contributing_members.clip(lower=1)
        ax.barh(yy, share, left=left, color=MEMBER_COLORS[column[2:]], height=.78, label=column[2:], lw=0)
        left += share.to_numpy()
    assert np.allclose(left, 1)
    names = [('★ ' if c else '') + (n[:36] + ('…' if len(n) > 36 else '')) for n, c in zip(frame.name, frame.core)]
    ax.set_yticks(yy, names, fontsize=5.1)
    ax.set(xlim=(0, 1), ylim=(len(frame) - .4, -.6), xlabel='share of contributing members (top 10% T_spatial)')
    bounds = np.flatnonzero(frame.category.to_numpy()[1:] != frame.category.to_numpy()[:-1]) + .5
    for b in bounds:
        ax.axhline(b, color=INK, lw=.6)
    for label, block in frame.groupby('category', sort=False):
        positions = yy[frame.category.eq(label).to_numpy()]
        ax.text(1.02, positions.mean(), f'{label}\n({len(block)})', transform=ax.get_yaxis_transform(), va='center',
                fontsize=5.3)
    ax.legend(loc='upper left', bbox_to_anchor=(-.75, -.07), ncol=6, fontsize=5.2, handlelength=.9, columnspacing=.6,
              borderaxespad=0)
    title(ax, f'{len(frame)} robust pathways: members by category (★ core)')
    letter(ax, 'd', dx=-140)
    ax = axes_mm(fig, 148, 58, 28, 30)
    totals = frame.category.value_counts().reindex(LABEL_ORDER).fillna(0).astype(int)
    colors = [MOUSE, CLASS_COLORS['conserved'], HUMAN, '#7F7F7F', '#C8C8C8']
    ax.barh(np.arange(len(totals))[::-1], totals.to_numpy(), color=colors, height=.6)
    for yi, value in zip(np.arange(len(totals))[::-1], totals.to_numpy()):
        ax.text(value + .4, yi, str(value), va='center', fontsize=5.5)
    ax.set_yticks(np.arange(len(totals))[::-1], ['mouse summary', 'shared', 'human-led', 'mixed', 'unresolved'], fontsize=5.5)
    ax.set(xlim=(0, max(totals.max(), 1) * 1.25), xlabel='pathways')
    title(ax, 'Pathway labels')
    save(fig, 'figure_3')


figure_3()

## 7 · Figure 4 · Robustness and agreement with independent data (R4)

- **a** Effect retained (AUC effect ÷ original) for the robust pathways under each saved check,
  with the share ≥ 0.7. The DPT row divides notebook 12's saved DPT effects by the scFates
  effects, as notebook 37 did. The green bar is the power-matched random removal of mouse S3
  (median 5th percentile to median).
- **b** Reviewed segment transitions on the coordinate, per specimen.
- **c** Gene-level agreement: our centred species × segment interaction against the Census
  (human − male mouse)(S3 − early) donor-level Welch t. The ρ values printed are notebook 37's
  saved summaries, computed on the centred contrast; that contrast itself is not saved.
- **d** Pathway-level replication z (donors as replicates). Lake/KPMP z for uncalled pathways is
  not saved, so only Lake's robust pathways are shown.
- **e** Specimen-level split-plot gene test, Q–Q plot. The relabeling p-values come from notebook
  37's stage cache; the species p-values in that cache are checked against the saved table.

In [ ]:
RETENTION_CHECKS = [('ratio_registration', 'anatomy-anchored coordinate'), ('ratio_cortical', 'cortical-like mouse S3 only'),
                    ('ratio_detected_in_both', 'genes detected in both species'), ('ratio_sex_unbiased', 'mouse sex-biased genes removed'),
                    ('ratio_flagged_removed', 'flagged genes removed'), ('ratio_matching_5bins', 'finer matching (5 bins)'),
                    ('ratio_probe_equal', 'equal probe counts'), ('dpt', 'DPT coordinate (notebooks 03, 12)')]
CHECK_RATIO = .7


def figure_4():
    F = 'Figure 4'
    s2 = table(F, 'pt_pathway_final/scfates/tables/table_S2_pathways.csv', index_col=0)
    dpt = table(F, 'pt_pathway_remodeling/pathway_spatial_evidence.csv')
    transitions = table(F, 'pt_pathway_final/scfates/tables/segment_transitions_by_specimen.csv')
    genes = table(F, 'pt_pathway_final/scfates/tables/gene_level_statistics.csv', index_col=0)
    summary = table(F, 'pt_pathway_final/scfates/tables/external_replication_summary.csv', index_col=0)
    s3 = table(F, 'pt_pathway_final/scfates/tables/table_S3_external_replication.csv', header=[0, 1], index_col=0)
    tests = table(F, 'pt_pathway_final/scfates/tables/matched_and_joint_tests_all_partitions.csv',
                  usecols=['pathway_id', 'statistic', 'partition', 'effect', 'q_normal'])
    split = {name: cached_arrays(F, 'pt_pathway_final/scfates', f'split_plot_{name}')
             for name in ('species', 'swap_HUK1_COR1_Ctrl1A2', 'swap_HUK1_COR1_Ctrl1A4')}
    np.testing.assert_allclose(split['species']['p'], genes.p_split.to_numpy(), rtol=1e-10)
    robust = s2[s2.robust.eq(True)]
    dpt = dpt[dpt.statistic.eq('T_spatial')].set_index('pathway_id')
    fig = new_figure(140)

    # a · retention
    ax = axes_mm(fig, 44, 20, 46, 108)
    core_color, robust_color = INK, '#A8A8A8'
    for yi, (column, label) in enumerate(RETENTION_CHECKS):
        values = dpt.effect.reindex(robust.index) / robust.effect if column == 'dpt' else robust[column]
        jitter = np.random.default_rng(yi).uniform(-.2, .2, len(values))
        ax.scatter(values, np.full(len(values), yi) + jitter, s=5, lw=0, c=np.where(robust.core.eq(True), core_color, robust_color))
        share = float((values >= CHECK_RATIO).mean())
        ax.text(1.02, yi, f'{share:.0%}', transform=ax.get_yaxis_transform(), va='center', fontsize=5.8)
        if column == 'ratio_cortical':
            band = robust[['power_ratio_p05', 'power_ratio_median']].median()
            ax.plot([band.power_ratio_p05, band.power_ratio_median], [yi + .38] * 2, color=ACCENT, lw=2.2, solid_capstyle='butt')
    ax.text(1.02, -.62, '≥ 0.7', transform=ax.get_yaxis_transform(), va='bottom', fontsize=5.8, fontweight='bold')
    ax.axvline(CHECK_RATIO, color=MUTED, ls='--', lw=.6)
    ax.axvline(1, color=LIGHT, lw=.6)
    ax.set_yticks(range(len(RETENTION_CHECKS)), [label for _, label in RETENTION_CHECKS])
    ax.set(ylim=(-.6, len(RETENTION_CHECKS) - .4), xlim=(.64, 1.32), xlabel='Effect retained (AUC effect ÷ original)')
    ax.invert_yaxis()
    handles = [Line2D([], [], marker='o', ls='', color=core_color, ms=3, label=f'core ({int(robust.core.eq(True).sum())})'),
               Line2D([], [], marker='o', ls='', color=robust_color, ms=3, label=f'robust only ({int(robust.core.eq(False).sum())})'),
               Line2D([], [], color=ACCENT, lw=2.2, label='power-matched removal')]
    ax.legend(handles=handles, loc='upper left', bbox_to_anchor=(-.9, -.07), ncol=3, fontsize=5.5, handletextpad=.3,
              columnspacing=.8)
    title(ax, f'Effect retention, {len(robust)} robust pathways')
    letter(ax, 'a', dx=-118)

    # b · segment transitions
    ax = axes_mm(fig, 106, 86, 26, 40)
    table_t = transitions.pivot(index='specimen', columns='transition', values='position')
    for name in SPECIMENS['mouse'] + SPECIMENS['human']:
        for yi, transition in enumerate(table_t.columns):
            ax.scatter(table_t.loc[name, transition], yi + (.12 if name in ('Ctrl1A4', 'HUK1_MED1') else -.12),
                       marker=SPECIMEN_MARKER[name], s=14, color=HUMAN if SPECIES_OF[name] == 'human' else MOUSE)
    ax.set_yticks(range(len(table_t.columns)), table_t.columns)
    ax.set(ylim=(-.7, 1.7), xlim=(.2, .7), xlabel='Position on coordinate')
    handles = [Line2D([], [], marker=SPECIMEN_MARKER[n], ls='', ms=3, color=HUMAN if SPECIES_OF[n] == 'human' else MOUSE,
                      label=SHORT[n]) for n in SPECIMENS['mouse'] + SPECIMENS['human']]
    ax.legend(handles=handles, loc='center', fontsize=5.5, ncol=2, handletextpad=.1, columnspacing=.4, borderaxespad=0)
    title(ax, 'Segment transitions')
    letter(ax, 'b', dx=-26)

    # c · gene-level agreement
    ax = axes_mm(fig, 146, 86, 32, 40)
    informative = genes[genes.informative_external.eq(True)]
    top = informative.T_spatial.ge(informative.T_spatial.quantile(.9))
    ax.scatter(informative.external_t_census_male[~top], informative.ours_interaction_centred[~top], s=1.5, lw=0, color=LIGHT,
               rasterized=True, label='informative genes')
    ax.scatter(informative.external_t_census_male[top], informative.ours_interaction_centred[top], s=2, lw=0, color=INK,
               rasterized=True, label='top 10% T_spatial')
    ax.axhline(0, color=NULL, lw=.4)
    ax.axvline(0, color=NULL, lw=.4)
    ax.set(xlabel='Census (human − male mouse)\n(S3 − early), Welch t', ylabel='Ours (human − mouse)(S3 − early)')
    ax.set_xlim(np.quantile(informative.external_t_census_male, [.001, .999]))
    ax.set_ylim(np.quantile(informative.ours_interaction_centred, [.001, .999]))
    primary_ref = 'Census cortex − male mouse (primary)'
    lake_ref = 'Lake/KPMP cortex − male mouse'
    rho_text = '\n'.join(f'{label}: ρ {summary.loc["gene ρ, top 10% T_spatial", ref]:.2f}\n  matched random '
                         f'{summary.loc["gene ρ, expression-matched random (median)", ref]:.2f}'
                         for label, ref in (('Census', primary_ref), ('Lake', lake_ref)))
    ax.text(.98, .02, rho_text, transform=ax.transAxes, va='bottom', ha='right', fontsize=5.2,
            bbox=dict(facecolor='white', edgecolor='none', alpha=.85, pad=1))
    ax.legend(loc='upper left', fontsize=5.2, markerscale=3, handletextpad=0, borderaxespad=0)
    title(ax, 'Gene-level agreement')
    letter(ax, 'c', dx=-26)

    # d · pathway replication
    ax = axes_mm(fig, 108, 20, 28, 44)
    swaps = [p for p in tests.partition.unique() if p.startswith('swap')]
    spatial = tests[tests.statistic.eq('T_spatial')]
    relabel_called = sorted(set(spatial.loc[spatial.partition.isin(swaps) & spatial.effect.gt(0) & spatial.q_normal.le(ALPHA),
                                            'pathway_id']))
    uncalled = s2.index[s2.candidate.eq(False)]
    groups = [(robust.external_z, f'Census\nrobust', HUMAN), (s2.loc[uncalled, 'external_z'].dropna(), 'Census\nuncalled', LIGHT),
              (s3.loc[robust.index, (lake_ref, 'z_joint')], 'Lake\nrobust', HUMAN),
              (s2.loc[relabel_called, 'external_z'].dropna(), f'Census\nrelabel-\ncalled', NULL)]
    np.testing.assert_allclose(robust.external_z.median(), summary.loc['median replication z, robust', primary_ref], rtol=1e-9)
    np.testing.assert_allclose(s3.loc[robust.index, (lake_ref, 'z_joint')].median(), summary.loc['median replication z, robust', lake_ref], rtol=1e-9)
    ax.boxplot([g[0] for g in groups], widths=.55, showfliers=False, medianprops=dict(color=INK, lw=1),
               boxprops=dict(lw=.6), whiskerprops=dict(lw=.6), capprops=dict(lw=.6))
    for k, (values, _, color) in enumerate(groups, start=1):
        many = len(values) > 200
        ax.scatter(np.full(len(values), k) + np.random.default_rng(k).uniform(-.15, .15, len(values)), values,
                   s=1.2 if many else 4, lw=0, alpha=.35 if many else .9, color=color, zorder=3, rasterized=many)
    ax.set_xticks(range(1, len(groups) + 1), [f'{g[1]}\n({len(g[0])})' for g in groups], fontsize=5.2)
    ax.axhline(0, color=NULL, lw=.5)
    ax.set_ylabel('Replication z (donors as replicates)')
    p_census = summary.loc['Mann–Whitney p, robust > uncalled', primary_ref]
    mantissa, exponent = f'{p_census:.0e}'.split('e')
    ax.text(.02, .98, f'robust > uncalled, Census:\n$p = {mantissa} \\times 10^{{{int(exponent)}}}$', transform=ax.transAxes,
            va='top', fontsize=5.2)
    title(ax, 'Pathway-level replication')
    letter(ax, 'd', dx=-26)

    # e · split-plot Q–Q
    ax = axes_mm(fig, 148, 20, 30, 44)
    from scipy.stats import f as f_distribution
    lambdas = {}
    for name, color, label in (('species', HUMAN, 'species'), ('swap_HUK1_COR1_Ctrl1A2', NULL, 'relabeling 1'),
                               ('swap_HUK1_COR1_Ctrl1A4', '#5A5A5A', 'relabeling 2')):
        p = np.clip(np.sort(split[name]['p']), 1e-300, 1)
        expected = (np.arange(1, len(p) + 1) - .5) / len(p)
        ax.plot(-np.log10(expected), -np.log10(p), color=color, lw=1, label=label)
        lambdas[name] = float(np.median(split[name]['F']) / f_distribution.median(3, float(split[name]['df_total'])))
    ax.plot([0, 4.3], [0, 4.3], color=MUTED, lw=.5, ls='--')
    ax.set(xlabel='Expected −log10 p', ylabel='Observed −log10 p', ylim=(0, 12), xlim=(0, 4.5))
    ax.legend(loc='upper right', fontsize=5.5, borderaxespad=0)
    worst = max(v for k, v in lambdas.items() if k != 'species')
    title(ax, f'Split-plot gene test\n(relabeling λ ≤ {worst:.2f})')
    letter(ax, 'e', dx=-26)
    save(fig, 'figure_4')
    return {'relabel_called': len(relabel_called), 'lambda': lambdas,
            'genes_p01': {k: float(np.mean(v['p'] <= .01)) for k, v in split.items()}}


FIGURE_4_DIAGNOSTICS = figure_4()
print(FIGURE_4_DIAGNOSTICS)

## 8 · Figure 5 · Lead genes against direct segment measurements (R5)

- *Left of each card:* human (orange) and mouse (blue) fitted curves along the PT coordinate,
  from notebook 38's stage cache and checked against its saved `T_spatial`. Each specimen's means
  in 12 coordinate bins come from section 3 (circles M1 and H1, squares M2 and H2). The shading
  marks each reviewed segment's 10th–90th percentile on the coordinate.
- *Right of each card:* each external dataset's S1, S2 and S3 values, centred on its own S1
  (log2):
  - Lake human cortex and mouse microdissection have all three segments.
  - Mouse snRNA by sex: segment means recomputed in section 3 with notebook 38's aggregation, after
    reproducing its saved S3 − S1 contrasts.
  - Rat is not shown: notebook 43 found its replicate reliability below the pre-registered
    threshold, and rat-based arguments are withdrawn.
- *Card titles:* gene, story, the pre-specified verdict, the symmetric zonation class (notebook
  43, union of S2 − S1 and S3c − early) and the probe counts (mouse/human).

The gene list below is the draft's current one. The coordinator may change it; each gene needs a
row in `story_gene_evidence.csv`.

In [ ]:
FIGURE_5_GENES = ['Gatm', 'Gamt', 'Acadm', 'Acaa2', 'Gclc', 'Ugt1a9', 'Ephx1', 'Cyp2e1', 'Dcxr']
STORY_NAMES = {'creatine': 'creatine synthesis', 'fao': 'fatty-acid oxidation', 'detox': 'drug handling',
               'reversal': 'reversal', 'sterol': 'sterol synthesis', 'lost_in_mouse': 'human-only expression',
               'mouse_only_lipid': 'mouse-only lipid', 'early_program_later': 'early program in late PT',
               'oat_illustration': 'organic anions', 'probe_example': 'probe example'}


def gene_class(gene, symmetric, one_species):
    """Symmetric class (notebook 43, union of S2 − S1 and S3c − early), else one-species expression (notebook 40)."""
    if gene in symmetric.index and isinstance(symmetric.loc[gene, 'union S2-S1 + S3c-early'], str):
        return symmetric.loc[gene, 'union S2-S1 + S3c-early']
    hit = one_species[one_species.gene.eq(gene)]
    return f'{hit.expressed_in.iloc[0]}-only expression' if len(hit) else 'not classifiable'


def segment_spans(figure):
    obs = read_pt_obs(figure)
    segment = obs.segment_class.astype(str).str.replace('PT-', '', regex=False)
    return {s: np.quantile(obs.shared_pseudospace[segment.eq(s)], [.1, .9]) for s in ('S1', 'S2', 'S3')}


def external_profiles(row, lake, external_means=None):
    """Each external dataset's segment values centred on S1 (log2); NaN where a segment is not saved."""
    profiles = {}
    if row.gene in lake.index:
        values = lake.loc[row.gene, ['lake_log2cpm_S1', 'lake_log2cpm_S2', 'lake_log2cpm_S3']].to_numpy(float)
        profiles['Lake human cortex'] = values - values[0]
    for sex in ('male', 'female'):
        tpm = np.array(str(row[f'micro_{sex}_tpm_S1_S2_S3']).split('/'), float)
        values = np.log2(tpm + 1)
        assert abs((values[2] - values[0]) - row[f'mouse_micro_{sex}']) < .06, (row.gene, sex)
        profiles[f'mouse microdissection, {sex}'] = values - values[0]
        profiles[f'mouse snRNA, {sex}'] = np.array([0, np.nan, row[f'mouse_sn_{sex}']])
    if external_means is not None:   # full S1/S2/S3 means (section 3) replace the S3 − S1-only segments
        for _, saved in external_means[external_means.gene.eq(row.gene)].iterrows():
            values = saved[['S1', 'S2', 'S3']].to_numpy(float)
            profiles[saved.dataset] = values - values[0]
    return profiles


PROFILE_STYLE = {'Lake human cortex': (HUMAN, '-', 'o', True), 'mouse microdissection, male': (MOUSE, '-', 's', True),
                 'mouse microdissection, female': (MOUSE, '--', 's', False), 'mouse snRNA, male': (MOUSE, ':', 'o', True),
                 'mouse snRNA, female': (MOUSE, (0, (1, 2)), 'o', False)}


def figure_5():
    F = 'Figure 5'
    evidence = table(F, 'pt_literature_deep/story_gene_evidence.csv')
    lake = table(F, 'pt_literature_deep/lake_cortex_segment_means.csv', index_col=0)
    symmetric = table(F, f'pt_revision_classes/{REVISION_TAG}/tables/gene_classes_symmetric.csv', index_col=0)
    one_species = table(F, 'pt_zonation_classes/tables/one_species_expressed_genes.csv')
    curves = cached_arrays(F, 'pt_literature_deep', 'story_gene_curves')
    np.testing.assert_allclose(curves['T_spatial'], evidence.T_spatial.to_numpy(), rtol=1e-6)
    gene_row = {g: i for i, g in enumerate(evidence.gene)}
    grid = curves['grid']
    spans = segment_spans(F)
    bins = AGG['bins']
    record_recomputed(F, 'specimen means in 12 coordinate bins')
    external_means = AGG['sn_segment_means']
    record_recomputed(F, 'mouse snRNA segment means per sex (Census), notebook 38 aggregation')
    fig = new_figure(178)
    for k, gene in enumerate(FIGURE_5_GENES):
        row = evidence.iloc[gene_row[gene]]
        col, rw = k % 3, k // 3
        x0, y0 = 11 + col * 57.5, 126 - rw * 55
        ax = axes_mm(fig, x0, y0, 28, 34)
        for j, s in enumerate(('S1', 'S2', 'S3')):
            ax.axvspan(*spans[s], color='#000000', alpha=.04 + .03 * j, lw=0, zorder=0)
            if rw == 0:
                ax.text(np.mean(spans[s]), 1.01, s, transform=ax.get_xaxis_transform(), ha='center', va='bottom',
                        fontsize=5.5, color=MUTED)
        ax.plot(grid, curves['human'][gene_row[gene]], color=HUMAN, lw=1.3)
        ax.plot(grid, curves['mouse'][gene_row[gene]], color=MOUSE, lw=1.3)
        draw_specimen_bins(ax, bins, gene)
        ax.set_xlim(grid[0], grid[-1])
        ax.set_xticks([.2, .4, .6, .8])
        if col == 0:
            ax.set_ylabel('log-norm expression')
        if rw == 2:
            ax.set_xlabel('PT coordinate')
        klass = gene_class(gene, symmetric, one_species)
        verdict = str(row.verdict)
        ax.annotate(f'{gene}', xy=(0, 1), xycoords='axes fraction', xytext=(0, 18 if rw == 0 else 13),
                    textcoords='offset points', fontsize=7, fontweight='bold', style='italic', va='bottom')
        ax.annotate(f'{STORY_NAMES.get(row.story, row.story)} · {verdict} · {klass} · probes M{int(row.mouse_probes)}/H{int(row.human_probes)}',
                    xy=(0, 1), xycoords='axes fraction', xytext=(0, 11 if rw == 0 else 6), textcoords='offset points',
                    fontsize=5.3, va='bottom', color=INK)
        ax2 = axes_mm(fig, x0 + 36, y0, 15, 34)
        for label, values in external_profiles(row, lake, external_means).items():
            color, style, marker, filled = PROFILE_STYLE[label]
            keep = np.isfinite(values)
            ax2.plot(np.arange(3)[keep], values[keep], color=color, ls=style, lw=.9, marker=marker, ms=2.6,
                     mfc=color if filled else 'white', mec=color, mew=.6)
        ax2.axhline(0, color=NULL, lw=.4)
        ax2.set_xticks([0, 1, 2], ['S1', 'S2', 'S3'])
        ax2.set_xlim(-.3, 2.3)
        ax2.tick_params(axis='y', labelsize=5.5)
        if col == 2:
            ax2.yaxis.set_label_position('right')
            ax2.set_ylabel('relative to S1 (log2)')
    handles = [Line2D([], [], color=HUMAN, lw=1.3, label='human, fitted (ours)'), Line2D([], [], color=MOUSE, lw=1.3, label='mouse, fitted (ours)')]
    handles += [Line2D([], [], marker='o', ls='', color=MUTED, ms=2.5, label='specimen bin means: M1, H1'),
                Line2D([], [], marker='s', ls='', color=MUTED, ms=2.5, label='specimen bin means: M2, H2')]
    handles += [Line2D([], [], color=c, ls=s, marker=m, ms=2.6, mfc=c if f else 'white', mec=c, label=l)
                for l, (c, s, m, f) in PROFILE_STYLE.items()]
    fig.legend(handles=handles, loc='lower center', ncol=5, fontsize=5.5, bbox_to_anchor=(.5, 0), columnspacing=1.0)
    save(fig, 'figure_5')


figure_5()

## 9 · Figure S1 · Coordinate limitations (notebooks 35 and 36)

- **a** Thinning every structure to equal depth: agreement with the original order per specimen and
  segment, and the correlation of position with library size before and after.
- **b** Human − mouse registration gap (largest segment shift) across refits, scFates and DPT.
- **c** Within-segment rank agreement between coordinates built from disjoint gene folds, and the
  numerical floor (a refit after a 10⁻¹⁶ input change).
- **d** Held-out gene prediction in the other specimen: gain over the segment and coverage oracles.
- **e** Attenuation: amplitudes of conserved held-out genes, and the early/late pathway split after
  adding positional noise to mouse. The shares of genes with smaller human amplitude are computed from
  the saved gene table; the coordinate share is checked against notebook 36's saved value.

In [ ]:
def specimen_points(ax, frame, value, x_of, jitter=.12):
    for name, block in frame.groupby('specimen'):
        xs = block.segment.map(x_of).to_numpy(float) + (jitter if name in ('Ctrl1A4', 'HUK1_MED1') else -jitter)
        ax.scatter(xs, block[value], marker=SPECIMEN_MARKER[name], s=11, lw=0,
                   color=HUMAN if SPECIES_OF[name] == 'human' else MOUSE, zorder=3, label=SHORT[name])


def figure_s1():
    F = 'Figure S1'
    p5 = table(F, 'pt_reconstruction_v2/nb35/p5_exposure_invariance.csv')
    gaps = table(F, 'pt_reconstruction_v2/nb35/species_registration_across_refits.csv')
    folds = table(F, 'pt_reconstruction_v2/nb35/p3a_gene_fold_within_segment_agreement.csv')
    floor = table(F, 'pt_reconstruction_v2/nb35/numerical_floor.csv')
    heldout = table(F, 'pt_reconstruction_v2/nb36/p4_heldout_prediction.csv')
    amplitudes = table(F, 'pt_reconstruction_v2/nb35/a3_conserved_heldout_amplitudes.csv')
    noise = table(F, 'pt_reconstruction_v2/nb35/a3b_noise_matching_summary.csv')
    split = table(F, 'pt_reconstruction_v2/nb36/early_late_split_by_coordinate.csv', index_col=0)
    x_of = {'S1': 0, 'S2': 1, 'S3': 2}
    fig = new_figure(178)
    scf = p5[p5.coordinate.eq('SCF13')]

    ax = axes_mm(fig, 14, 128, 34, 38)
    specimen_points(ax, scf, 'rho(equal-depth, original)', x_of)
    ax.set_xticks([0, 1, 2], ['S1', 'S2', 'S3'])
    ax.set(xlim=(-.5, 2.5), ylim=(0, 1), ylabel='ρ, equal-depth vs original order')
    ax.legend(loc='lower right', fontsize=5.5, ncol=2, handletextpad=.1, columnspacing=.4, borderaxespad=0)
    title(ax, 'Equal-depth refit (scFates)')
    letter(ax, 'a', dx=-26)
    ax = axes_mm(fig, 60, 128, 34, 38)
    for _, row in scf.iterrows():
        x = x_of[row.segment] + (.12 if row.specimen in ('Ctrl1A4', 'HUK1_MED1') else -.12)
        color = HUMAN if row.species == 'human' else MOUSE
        ax.annotate('', xy=(x, row['rho(z, log library) after']), xytext=(x, row['rho(z, log library) before']),
                    arrowprops=dict(arrowstyle='-|>', color=color, lw=.7, mutation_scale=5))
        ax.scatter(x, row['rho(z, log library) before'], s=8, color=color, marker=SPECIMEN_MARKER[row.specimen], lw=0)
    ax.axhline(0, color=NULL, lw=.5)
    ax.set_xticks([0, 1, 2], ['S1', 'S2', 'S3'])
    ax.set(xlim=(-.5, 2.5), ylabel='ρ(position, log library size)')
    title(ax, 'Depth dependence: before → after')

    ax = axes_mm(fig, 128, 128, 48, 38)
    order = list(dict.fromkeys(gaps.refit))
    for coordinate, color, dy in (('SCF13', INK, -.15), ('DPT13', NULL, .15)):
        block = gaps[gaps.coordinate.eq(coordinate)].set_index('refit').reindex(order)
        ax.scatter(block['max |gap|'], np.arange(len(order)) + dy, s=12, color=color, label='scFates' if coordinate == 'SCF13' else 'DPT',
                   zorder=3)
    ax.set_yticks(range(len(order)), order)
    ax.invert_yaxis()
    ax.set(xlabel='largest human − mouse segment shift', xlim=(0, None))
    ax.legend(loc='upper right', fontsize=5.5, borderaxespad=0)
    title(ax, 'Registration gap across refits')
    letter(ax, 'b', dx=-64)

    ax = axes_mm(fig, 14, 72, 40, 38)
    for coordinate, color, dx in (('SCF13', INK, -.15), ('DPT13', NULL, .15)):
        block = folds[folds.coordinate.eq(coordinate)]
        xs = block.segment.map(x_of) + dx + np.random.default_rng(1).uniform(-.06, .06, len(block))
        ax.scatter(xs, block.rho, s=5, lw=0, color=color, label='scFates' if coordinate == 'SCF13' else 'DPT')
        medians = block.groupby('segment').rho.median()
        for seg, value in medians.items():
            ax.plot([x_of[seg] + dx - .12, x_of[seg] + dx + .12], [value] * 2, color=color, lw=1.4)
    ax.set_xticks([0, 1, 2], ['S1', 'S2', 'S3'])
    ax.set(xlim=(-.5, 2.5), ylim=(-.65, 1), ylabel='within-segment ρ between gene folds')
    ax.axhline(0, color=NULL, lw=.4)
    ax.legend(loc='lower left', fontsize=5.5, borderaxespad=0)
    title(ax, 'Gene-fold agreement (bars: median)')
    letter(ax, 'c', dx=-26)
    ax = axes_mm(fig, 66, 72, 34, 38)
    for coordinate, color, dx in (('SCF13', INK, -.15), ('DPT13', NULL, .15)):
        block = floor[floor.coordinate.eq(coordinate)]
        ax.scatter(block.segment.map(x_of) + dx, block['max |Δz|'], s=8, lw=0, color=color)
    ax.set_xticks([0, 1, 2], ['S1', 'S2', 'S3'])
    ax.set(xlim=(-.5, 2.5), ylabel='largest position change')
    ax.set_yscale('log')
    title(ax, 'Numerical floor (10⁻¹⁶ input change)')

    ax = axes_mm(fig, 124, 72, 52, 38)
    arms = list(dict.fromkeys(heldout.arm))
    for column, marker, dx in (('gain vs segment oracle', 'o', -.13), ('gain vs quadratic coverage oracle', 'D', .13)):
        for yi, arm in enumerate(arms):
            block = heldout[heldout.arm.eq(arm)].dropna(subset=[column])
            ax.scatter(block[column], np.full(len(block), yi + dx), marker=marker, s=7, lw=0,
                       color=[HUMAN if s == 'human' else MOUSE for s in block.species])
    ax.axvline(0, color=MUTED, lw=.6)
    ax.set_yticks(range(len(arms)), arms)
    ax.invert_yaxis()
    ax.set_xlabel('gain in held-out prediction (R²)')
    handles = [Line2D([], [], marker='o', ls='', color=INK, ms=2.5, label='vs segment oracle'),
               Line2D([], [], marker='D', ls='', color=INK, ms=2.5, label='vs coverage oracle'),
               Line2D([], [], marker='s', ls='', color=MOUSE, ms=2.5, label='mouse'),
               Line2D([], [], marker='s', ls='', color=HUMAN, ms=2.5, label='human')]
    ax.legend(handles=handles, loc='upper left', bbox_to_anchor=(-.05, -.2), ncol=2, fontsize=5.5, borderaxespad=0,
              columnspacing=.8)
    title(ax, 'Held-out genes, other specimen')
    letter(ax, 'd', dx=-40)

    ax = axes_mm(fig, 14, 16, 36, 38)
    attenuation = table(F, 'pt_reconstruction_v2/nb36/attenuation_by_arm.csv', index_col=0)
    shares = {'coordinate': float((amplitudes.amp_human < amplitudes.amp_mouse).mean()),
              'labels': float((amplitudes.label_amp_human < amplitudes.label_amp_mouse).mean()),
              'external': float(amplitudes.external_human_smaller.mean())}
    np.testing.assert_allclose(shares['coordinate'], attenuation.loc['SCF13', 'fraction human < mouse'], rtol=1e-9)
    lim = float(np.nanmax(np.abs(amplitudes[['amp_mouse', 'amp_human', 'label_amp_mouse', 'label_amp_human']].to_numpy()))) * 1.08
    ax.scatter(amplitudes.label_amp_mouse, amplitudes.label_amp_human, s=7, facecolor='white', edgecolor=NULL, lw=.6,
               label='segment labels')
    ax.scatter(amplitudes.amp_mouse, amplitudes.amp_human, s=7, lw=0, color=INK, label='coordinate')
    ax.plot([-lim, lim], [-lim, lim], color=MUTED, lw=.6, ls='--')
    ax.legend(loc='upper left', fontsize=5.2, borderaxespad=0, handletextpad=.1)
    ax.text(.98, .03, 'human < mouse:\n' + '\n'.join(f'{k} {v:.0%}' for k, v in shares.items()), transform=ax.transAxes,
            ha='right', va='bottom', fontsize=5.2)
    ax.axhline(0, color=NULL, lw=.4)
    ax.axvline(0, color=NULL, lw=.4)
    ax.set(xlim=(-lim, lim), ylim=(-lim, lim), xlabel='mouse amplitude (S3 − S1)', ylabel='human amplitude')
    title(ax, f'{len(amplitudes)} conserved held-out genes')
    letter(ax, 'e', dx=-26)
    ax = axes_mm(fig, 66, 16, 40, 38)
    dose = noise[noise.source.eq('dose')].sort_values('sigma')
    base = split.loc['SCF13']
    sigma = np.r_[0, dose.sigma]
    for column, base_column, color, label in (('mouse-high pathways', 'mouse-high', MOUSE, 'mouse-high at peak'),
                                              ('human-high pathways', 'human-high', HUMAN, 'human-high at peak')):
        ax.plot(sigma, np.r_[base[base_column], dose[column]], marker='o', ms=3, color=color, label=label)
    for row in noise[noise.source.ne('dose')].itertuples():
        ax.axvline(row.sigma, color=MUTED, lw=.6, ls=':')
        ax.text(row.sigma, 60, f' estimated\n ({row.source})', fontsize=5.2, color=MUTED, va='top')
    ax.set(xlabel='noise SD added to mouse positions', ylabel='robust pathways (earlier list of 66)', ylim=(0, 66))
    ax.legend(loc='center left', fontsize=5.5, borderaxespad=0, bbox_to_anchor=(0, .55))
    title(ax, 'Early/late split after equal noise')
    save(fig, 'figure_S1')


figure_s1()

## 10 · Figure S2 · DPT specificity diagnostic (descriptive)

- **a** What the significant pathways are, by category, for the species split and the relabelings.
- **b** Within-species positional pathway calls on each coordinate.
- **c** Pooled within-segment spread (IQR) of each coordinate.

In [ ]:
def figure_s2():
    F = 'Figure S2'
    categories = table(F, 'pt_reconstruction_v2/dpt_diagnostic/call_categories.csv')
    within = table(F, 'pt_reconstruction_v2/dpt_diagnostic/within_species_tspatial.csv')
    placement = table(F, 'pt_reconstruction_v2/dpt_diagnostic/specimen_placement.csv')
    names = {'DPT13': 'DPT', 'SCF13': 'scFates'}
    fig = new_figure(70)
    ax = axes_mm(fig, 26, 22, 44, 38)
    columns = [c for c in categories.columns if c not in ('coordinate', 'kind')]
    shades = dict(zip(columns, ['#4D4D4D', '#9A9A9A', '#C8C8C8', '#E6E6E6']))
    short = {c: c.replace(' (metabolic, transport, signalling)', ': metabolic, transport, signalling')
             .replace(' / mitochondrial', '').replace(' / housekeeping', ', housekeeping') for c in columns}
    yy = np.arange(len(categories))[::-1]
    left = np.zeros(len(categories))
    for column in columns:
        ax.barh(yy, categories[column], left=left, color=shades[column], height=.6, label=short[column], edgecolor='white', lw=.4)
        left += categories[column].to_numpy()
    for yi, total in zip(yy, left):
        ax.text(total + 2, yi, str(int(total)), va='center', fontsize=5.5)
    ax.set_yticks(yy, [f'{names[c]}, {k}' for c, k in zip(categories.coordinate, categories.kind)])
    ax.set(xlabel='pathway calls (both relabelings summed)', xlim=(0, left.max() * 1.15))
    ax.legend(loc='upper left', bbox_to_anchor=(-.5, -.3), ncol=4, fontsize=5.2, handlelength=.9, columnspacing=.8,
              borderaxespad=0)
    title(ax, 'Categories of called pathways')
    letter(ax, 'a', dx=-60)
    ax = axes_mm(fig, 90, 22, 34, 38)
    x = np.arange(2)
    for k, (coordinate, color) in enumerate((('DPT13', NULL), ('SCF13', INK))):
        block = within[within.coordinate.eq(coordinate)].set_index('species').reindex(['mouse', 'human'])
        ax.bar(x + (k - .5) * .38, block['within-species pathway calls'], .36, color=color, label=names[coordinate])
        for xi, value in zip(x, block['within-species pathway calls']):
            ax.text(xi + (k - .5) * .38, value + .6, str(int(value)), ha='center', fontsize=5.5)
    ax.set_xticks(x, ['mouse vs mouse', 'human vs human'])
    ax.set(ylabel='positional pathway calls')
    ax.legend(loc='upper right', fontsize=5.5, borderaxespad=0)
    title(ax, 'Within-species calls')
    letter(ax, 'b', dx=-26)
    ax = axes_mm(fig, 140, 22, 36, 38)
    groups = [(sp, seg) for sp in ('mouse', 'human') for seg in ('S1', 'S2', 'S3')]
    for k, (coordinate, color) in enumerate((('DPT13', NULL), ('SCF13', INK))):
        block = placement[placement.coordinate.eq(coordinate)].set_index(['species', 'segment'])
        values = [block.loc[g, 'segment IQR (pooled)'] for g in groups]
        ax.bar(np.arange(len(groups)) + (k - .5) * .38, values, .36, color=color, label=names[coordinate])
    ax.set_xticks(range(len(groups)), [seg for sp, seg in groups])
    for k, species in enumerate(('mouse', 'human')):
        ax.text(1 + 3 * k, -.16, species, transform=blended_transform_factory(ax.transData, ax.transAxes), ha='center',
                va='top', fontsize=6)
    ax.set_ylabel('pooled within-segment IQR')
    ax.set_ylim(0, .3)
    ax.legend(loc='upper left', fontsize=5.5, borderaxespad=0, ncol=2)
    title(ax, 'Within-segment spread')
    letter(ax, 'c', dx=-26)
    save(fig, 'figure_S2')


figure_s2()

## 11 · Figure S3 · Position-dependent pathways (notebook 37)

- **a** Joint-test funnel.
- **b** Median member Δ(s) (human − mouse, log-normalized) of the robust pathways, grouped into
  programs, with peak ticks. Δ(s) is the median of notebook 37's fitted gene curves (stage cache,
  checked against the saved `T_spatial`) over each pathway's members. Membership comes from
  notebook 31's saved gene sets, checked against the saved member counts. A row is drawn only if
  its shape class matches notebook 37's saved one. Annotations: program, external replication
  (Census) and the split-plot matched test.
- **c** Peak position against the median member Z at the peak.
- **d** The six largest programs: member Δ(s) (grey) and program mean (colour = direction at the
  peak). Below, the fitted curves (orange human, blue mouse) and each specimen's means in 12 bins
  (section 3) for a direction-matched driver. This is the first shared driver, in notebook 37's
  T_spatial order, whose fitted Δ at the program's median peak has the program's sign.
- **e** Averaging loss by shape class.

In [ ]:
def robust_curves(figure, s2):
    """Median member Δ(s) and Z at the peak for the robust pathways, from saved fitted curves."""
    genes = table(figure, 'pt_pathway_final/scfates/tables/gene_level_statistics.csv', index_col=0)
    fit = cached_arrays(figure, 'pt_pathway_final/scfates', 'species_fit')
    np.testing.assert_allclose(fit['T_spatial'], genes.T_spatial.to_numpy(), rtol=1e-6, atol=1e-8)
    path = results_root / 'pt_pathway_method_selection' / 'proto_gene_sets.json'
    _record(figure, path, 'gene sets (notebook 31)')
    gene_sets = json.loads(path.read_text())
    index = pd.Index(genes.index)
    robust = s2.index[s2.robust.eq(True)]
    members = {p: [g for g in gene_sets[p] if g in index] for p in robust}
    sizes = pd.Series({p: len(v) for p, v in members.items()})
    assert sizes.equals(s2.loc[robust, 'n_tested'].astype(int).rename(None)), 'gene-set sizes differ from notebook 37'
    delta = pd.DataFrame(fit['delta'], index=index)
    z = pd.DataFrame(fit['z'], index=index)
    median_delta = pd.DataFrame({p: delta.loc[members[p]].median() for p in robust}).T
    from pseudospace.pathway_decoys import shape_class
    shapes = pd.Series({p: shape_class(median_delta.loc[p].to_numpy()) for p in robust})
    assert shapes.equals(s2.loc[robust, 'shape_class'].rename(None)), 'shape classes differ from notebook 37'
    grid = fit['grid']
    peak_index = {p: int(np.argmin(np.abs(grid - s2.loc[p, 'peak_position']))) for p in robust}
    z_at_peak = pd.Series({p: float(z.loc[members[p]].iloc[:, peak_index[p]].median()) for p in robust})
    assert (z_at_peak.gt(0) == s2.loc[robust, 'human_high_at_peak'].astype(bool)).all(), 'direction at peak differs'
    return median_delta, z_at_peak, grid, fit, index


def direction_matched_driver(program, fit, gene_position, grid):
    """First listed driver (drivers are ordered by T_spatial) whose fitted Δ at the program's median peak has the
    program's sign; falls back to the first driver, flagged, when none matches."""
    peak = int(np.argmin(np.abs(grid - program.median_peak)))
    sign = 1 if program.direction == 'human-high' else -1
    drivers = program.drivers.split(', ')
    for gene in drivers:
        if gene in gene_position and np.sign(fit['delta'][gene_position[gene]][peak]) == sign:
            return gene, True
    return drivers[0], False


def figure_s3():
    F = 'Figure S3'
    s2 = table(F, 'pt_pathway_final/scfates/tables/table_S2_pathways.csv', index_col=0)
    funnel = table(F, 'pt_pathway_final/scfates/tables/funnel.csv', index_col=0)['pathways']
    programs = table(F, 'pt_pathway_final/scfates/tables/table_S4_programs.csv', index_col=0)
    median_delta, z_at_peak, grid, fit, index = robust_curves(F, s2)
    robust = s2[s2.robust.eq(True)].copy()
    robust['program_size'] = robust.program.map(robust.program.value_counts())
    order = robust.sort_values(['program_size', 'program', 'peak_position'], ascending=[False, True, True]).index
    spans = segment_spans(F)
    fig = new_figure(238)

    # b · heatmap
    ax = axes_mm(fig, 60, 108, 52, 88)
    rows = median_delta.loc[order]
    limit = float(np.quantile(np.abs(rows.to_numpy()), .98))
    image = ax.imshow(rows.clip(-limit, limit).to_numpy(), aspect='auto', cmap='RdBu_r', vmin=-limit, vmax=limit,
                      extent=(grid[0], grid[-1], len(rows), 0), interpolation='nearest')
    ax.scatter(robust.loc[order, 'peak_position'], np.arange(len(order)) + .5, marker='|', s=18, color=INK, lw=.9)
    names = [('★ ' if robust.loc[p, 'core'] else '') + (robust.loc[p, 'name'][:42] + ('…' if len(robust.loc[p, 'name']) > 42 else ''))
             for p in order]
    ax.set_yticks(np.arange(len(order)) + .5, names, fontsize=5.2)
    program_values = robust.loc[order, 'program'].to_numpy()
    for b in np.flatnonzero(program_values[1:] != program_values[:-1]) + 1:
        ax.axhline(b, color='white', lw=1.4)
    for s, (a_q, b_q) in spans.items():
        ax.text((a_q + b_q) / 2, -.4, s, ha='center', va='bottom', fontsize=5.5, color=MUTED)
    ax.set_xlabel('PT coordinate')
    cax = axes_mm(fig, 66, 95, 40, 1.6)
    fig.colorbar(image, cax=cax, orientation='horizontal')
    cax.set_xlabel('median member Δ(s), human − mouse (log-norm)', fontsize=5.5)
    cax.tick_params(labelsize=5.5)
    title(ax, f'{len(order)} robust pathways in {robust.program.nunique()} programs (★ core; tick = peak)', pad=11)
    letter(ax, 'b', dx=-170, dy=8)
    ax = axes_mm(fig, 113.5, 108, 10, 88)
    external = robust.loc[order, 'external_replication'].map({'replicates': 1.0, 'not shown': .45, 'not testable': 0}).fillna(0)
    annotation = np.column_stack([(program_values % 2) * .3 + .1, external, robust.loc[order, 'split_plot_matched_q'].le(ALPHA) * 1.0])
    ax.imshow(annotation, aspect='auto', cmap='Greys', vmin=0, vmax=1.1, extent=(0, 3, len(order), 0), interpolation='nearest')
    for yi, p in enumerate(order):
        ax.text(.5, yi + .5, str(int(robust.loc[p, 'program'])), ha='center', va='center', fontsize=4.6, color=INK)
    ax.set_xticks([.5, 1.5, 2.5], ['program', 'external', 'split-plot'], rotation=90, fontsize=5.5)
    ax.set_yticks([])
    ax.tick_params(length=0)
    for side in ax.spines.values():
        side.set_visible(False)
    fig.text(132 / WIDTH_MM, 108 / fig.height_mm, 'b, annotations. external: black = replicates\n(Census), grey = not shown, white = not\n'
             'testable. split-plot: black = matched q ≤ 0.05', fontsize=5.2, va='top', color=MUTED)

    # a · funnel
    ax = axes_mm(fig, 60, 216, 40, 16)
    stages = funnel.iloc[1:]
    yy = np.arange(len(stages))[::-1]
    ax.barh(yy, stages.values, color=[LIGHT] * 3 + ['#7A7A7A', INK], height=.62)
    for y_value, value in zip(yy, stages.values):
        ax.text(value + 1, y_value, str(int(value)), va='center', fontsize=5.5)
    ax.set_yticks(yy, [s.replace(' (registration + region)', '').replace(' (+ shared-gene, sex-unbiased)', '') for s in stages.index],
                  fontsize=5.5)
    ax.set(xlabel=f'pathways (of {int(funnel["tested"]):,} tested)', xlim=(0, stages.max() * 1.2))
    title(ax, 'Joint-test funnel')
    letter(ax, 'a', dx=-170)

    # c · peak versus median member Z
    ax = axes_mm(fig, 140, 164, 36, 30)
    human_high = robust.human_high_at_peak.astype(bool)
    ax.scatter(robust.peak_position, z_at_peak.reindex(robust.index), s=9, lw=0, c=np.where(human_high, HUMAN, MOUSE))
    ax.axhline(0, color=NULL, lw=.5)
    for flag, color, va in ((True, HUMAN, 'bottom'), (False, MOUSE, 'top')):
        block = robust.loc[human_high.eq(flag), 'peak_position']
        ax.text(.98, .97 if flag else .03, f'{"human" if flag else "mouse"}-high: {len(block)}, median peak {block.median():.2f}',
                transform=ax.transAxes, ha='right', va='top' if flag else 'bottom', fontsize=5.3, color=INK)
    ax.set(xlabel='peak position on PT coordinate', ylabel='median member Z at peak', xlim=(0, 1))
    title(ax, 'Peak position by direction')
    letter(ax, 'c', dx=-26)

    # e · averaging loss
    ax = axes_mm(fig, 140, 124, 36, 22)
    shapes = ['graded', 'localized', 'reversing']
    for k, klass in enumerate(shapes):
        values = robust.loc[robust.shape_class.eq(klass), 'averaging_loss']
        ax.scatter(np.full(len(values), k) + np.random.default_rng(k).uniform(-.15, .15, len(values)), values, s=7, lw=0,
                   c=np.where(robust.loc[values.index, 'human_high_at_peak'].astype(bool), HUMAN, MOUSE))
        ax.text(k, .9, f'n = {len(values)}', ha='center', fontsize=5.5)
    ax.set_xticks(range(3), shapes)
    ax.set(ylim=(-.05, 1), ylabel='averaging loss')
    title(ax, 'Averaging loss by shape')
    letter(ax, 'e', dx=-26)

    # d · six largest programs
    bins = AGG['bins']
    record_recomputed(F, 'specimen means in 12 coordinate bins (driver genes)')
    human_symbols = table(F, 'minimal_pt_scfates/ortholog_map_used.csv').set_index('mouse_symbol').human_symbol.to_dict()
    top_programs = programs.sort_values('n_pathways', ascending=False, kind='stable').head(6).index
    gene_position = {g: i for i, g in enumerate(index)}
    for k, program in enumerate(top_programs):
        ids = robust.index[robust.program.eq(program)]
        x0 = 12 + k * 28.5
        ax = axes_mm(fig, x0, 52, 23, 24)
        for p in ids:
            ax.plot(grid, median_delta.loc[p], color='#BDBDBD', lw=.6)
        direction = programs.loc[program, 'direction']
        ax.plot(grid, median_delta.loc[ids].mean(), color=HUMAN if direction == 'human-high' else MOUSE, lw=1.3)
        ax.axhline(0, color=NULL, lw=.4)
        ax.set_xticks([.2, .5, .8])
        ax.tick_params(labelsize=5.3)
        name = programs.loc[program, 'name']
        ax.set_title(f'P{program} · {name[:17]}' + ('…' if len(name) > 17 else '') +
                     f'\n{int(programs.loc[program, "n_pathways"])} pathways, {direction}', loc='left', fontsize=5.6)
        if k == 0:
            ax.set_ylabel('Δ(s)')
            letter(ax, 'd', dx=-24, dy=10)
        driver, matched = direction_matched_driver(programs.loc[program], fit, gene_position, grid)
        ax = axes_mm(fig, x0, 14, 23, 24)
        i = gene_position[driver]
        ax.plot(grid, fit['human'][i], color=HUMAN, lw=1.2)
        ax.plot(grid, fit['mouse'][i], color=MOUSE, lw=1.2)
        draw_specimen_bins(ax, bins, driver)
        ax.set_xticks([.2, .5, .8])
        ax.tick_params(labelsize=5.3)
        ax.set_xlabel('PT coordinate', fontsize=5.5)
        human_symbol = human_symbols.get(driver, driver.upper())
        shown = driver if human_symbol == driver.upper() else f'{driver} / {human_symbol}'
        ax.set_title(f'driver: {shown}' + ('' if matched else ' (no direction match)'), loc='left', fontsize=5.8, style='italic')
        if k == 0:
            ax.set_ylabel('fitted log-norm')
            ax.legend(handles=[Line2D([], [], color=HUMAN, label='human'), Line2D([], [], color=MOUSE, label='mouse')],
                      loc='upper right', fontsize=5.2, borderaxespad=0, handlelength=1)
    save(fig, 'figure_S3')


figure_s3()

## 12 · Figure S4 · Zonation amplitude (notebook 39)

- **a** Human ÷ mouse amplitude against each independent reference: OLS with 95% interval (●),
  Deming (◇), orthogonal (□); external-only ratios in green. Dashed line: the primary ratio.
- **b, c** Both species on male microdissection and on Lake cortex: per-gene S2 − S1 gradients from
  section 3, which reproduce notebook 39's slopes and gene counts exactly. The lines are the
  saved OLS slopes.
- **d** Measurement checks. **e** Tissue state: AKI ÷ control, and low- and high-injury halves.
- **f** Lake donors: injury score against S2 − S1 amplitude. ρ is the Spearman correlation of the
  two saved columns.
- **g** Lead genes against global flattening. **h, i** Post-hoc reference-free spread ratios and
  disattenuated correlations.

In [ ]:
REFERENCE_STYLE = {'microdissection': (MOUSE, 'o', True), 'mouse snRNA': (MOUSE, 'o', False), 'human snRNA': (HUMAN, 'o', True)}


def figure_s4():
    F = 'Figure S4'
    ratios = table(F, 'pt_zonation_amplitude/tables/amplitude_ratios.csv')
    external = table(F, 'pt_zonation_amplitude/tables/external_amplitude_ratios.csv')
    checks = table(F, 'pt_zonation_amplitude/tables/measurement_checks.csv')
    aki = table(F, 'pt_zonation_amplitude/tables/aki_amplitude_ratios.csv')
    halves = table(F, 'pt_zonation_amplitude/tables/injury_halves.csv')
    donors = table(F, 'pt_zonation_amplitude/tables/lake_donor_scores.csv', index_col=0)
    lead = table(F, 'pt_zonation_amplitude/tables/lead_gene_flattening_classes.csv', index_col=0)
    posthoc = table(F, 'pt_zonation_amplitude/tables/posthoc_reference_free_amplitude.csv', index_col=0)
    decisions = table(F, 'pt_zonation_amplitude/tables/decisions.csv', index_col=0)['0']
    gradients = AGG['gradients']
    record_recomputed(F, 'per-gene S2 − S1 gradients: ours (reviewed labels), male microdissection, Lake healthy cortex')
    r = float(decisions[[k for k in decisions.index if k.startswith('primary ratio r')][0]])
    np.testing.assert_allclose(r, ratios[(ratios.contrast == 'S2-S1') & ratios.reference.eq('microdissection male (GSE150338)')].ratio_ols.iloc[0], atol=5e-4)
    fig = new_figure(212)

    # a · forest
    ax = axes_mm(fig, 50, 146, 42, 58)
    labels, y_value, positions = [], 0, []
    for contrast in ('S2-S1', 'S3c-early', 'S3-early'):
        for row in ratios[ratios.contrast.eq(contrast)].itertuples():
            color, marker, filled = REFERENCE_STYLE[row.group]
            ax.plot([row.ci_low, row.ci_high], [y_value] * 2, color=color, lw=1.1)
            ax.scatter(row.ratio_ols, y_value, s=12, color=color if filled else 'white', edgecolor=color, lw=.8, zorder=3)
            if np.isfinite(row.ratio_deming):
                ax.scatter(row.ratio_deming, y_value, marker='D', s=8, facecolor='white', edgecolor=INK, lw=.5, zorder=3)
            ax.scatter(row.ratio_orthogonal, y_value, marker='s', s=7, facecolor='white', edgecolor=MUTED, lw=.5, zorder=3)
            labels.append(f'{contrast} · {row.reference.replace(" (GSE150338)", " a").replace(" (GSE212213)", " b").replace(" (Census)", "")}')
            positions.append(y_value)
            y_value -= 1
        y_value -= .6
    for row in external.itertuples():
        ax.plot([row.ci_low, row.ci_high], [y_value] * 2, color=ACCENT, lw=1.1)
        ax.scatter(row.ratio_ols, y_value, s=12, color=ACCENT, zorder=3)
        labels.append(f'{row.contrast} · {row.analysis.split(",")[0].replace("Lake healthy ÷ ", "Lake ÷ ")} (external)')
        positions.append(y_value)
        y_value -= 1
    ax.set_yticks(positions, labels, fontsize=5)
    ax.set_xscale('log')
    ax.set_xticks([.1, .2, .5, 1, 2, 5], ['0.1', '0.2', '0.5', '1', '2', '5'])
    ax.axvline(r, color=MUTED, ls='--', lw=.6)
    ax.axvline(1, color=INK, lw=.5)
    ax.set_xlabel('human ÷ mouse amplitude (slope ratio)')
    handles = [Line2D([], [], marker='o', ls='-', color=MOUSE, ms=3, label='mouse reference axis'),
               Line2D([], [], marker='o', ls='-', color=HUMAN, ms=3, label='human reference axis'),
               Line2D([], [], marker='o', ls='-', color=ACCENT, ms=3, label='external only'),
               Line2D([], [], marker='D', ls='', mfc='white', mec=INK, ms=2.5, label='Deming'),
               Line2D([], [], marker='s', ls='', mfc='white', mec=MUTED, ms=2.5, label='orthogonal')]
    ax.legend(handles=handles, loc='upper left', bbox_to_anchor=(-1.05, -.09), ncol=3, fontsize=5.2, columnspacing=.6,
              handlelength=1.4)
    title(ax, f'Amplitude ratio (dashed: r = {r:.2f})')
    letter(ax, 'a', dx=-128)
    for k, (panel, name, reference) in enumerate((('b', 'male microdissection', 'microdissection male (GSE150338)'),
                                                  ('c', 'Lake human cortex', 'Lake healthy human cortex'))):
        ax = axes_mm(fig, 104 + k * 38, 156, 33, 46)
        row = ratios[ratios.contrast.eq('S2-S1') & ratios.reference.eq(reference)].iloc[0]
        x = gradients[reference].dropna()
        for species, color, slope in (('mouse', MOUSE, row.slope_mouse), ('human', HUMAN, row.slope_human)):
            y = gradients.loc[x.index, f'ours {species}']
            ax.scatter(x, y, s=1.5, lw=0, alpha=.3, color=color, rasterized=True)
            xs = np.quantile(x, [.005, .995])
            ax.plot(xs, y.mean() + slope * (xs - x.mean()), color=color, lw=1.2, label=f'our {species}: slope {slope:.2f}')
        ax.axhline(0, color=NULL, lw=.4)
        ax.axvline(0, color=NULL, lw=.4)
        ax.set(xlabel=f'{name}\nS2 − S1 (log2)', ylabel='our S2 − S1 (log2)')
        ax.legend(loc='upper left', fontsize=5.2, borderaxespad=0)
        title(ax, f'Both species on\n{name}')
        letter(ax, panel, dx=-10)

    # d · measurement checks
    ax = axes_mm(fig, 14, 90, 46, 34)
    analyses = list(dict.fromkeys(checks.analysis))
    references = list(dict.fromkeys(checks.reference))
    for k, reference in enumerate(references):
        block = checks[checks.reference.eq(reference)].set_index('analysis').reindex(analyses)
        color, marker, filled = REFERENCE_STYLE[block.group.iloc[0]]
        xs = np.arange(len(analyses)) + (k - 1) * .22
        short = {'microdissection male': 'mouse micro ♂', 'mouse snRNA male': 'mouse snRNA ♂', 'Lake healthy human cortex': 'Lake human'}
        ax.errorbar(xs, block.ratio_ols, yerr=[block.ratio_ols - block.ci_low.clip(lower=1e-2), block.ci_high - block.ratio_ols],
                    fmt='o', ms=2.8, color=color, mfc=color if filled else 'white', lw=.8,
                    label=short.get(reference.split(' (')[0], reference))
    ax.set_yscale('log')
    ax.set_ylim(.02, 400)
    ax.set_xticks(range(len(analyses)), [a.replace('human expression tertile: ', 'tertile ').replace('per-structure ', '')
                                         .replace('middle', 'mid').replace('equal probe counts', 'equal probes') for a in analyses],
                  rotation=45, ha='right', fontsize=5.5)
    ax.axhline(r, color=MUTED, ls='--', lw=.6)
    ax.axhline(1, color=INK, lw=.5)
    ax.set_ylabel('human ÷ mouse (S2 − S1)')
    ax.legend(loc='upper left', fontsize=5.2, borderaxespad=0, ncol=2, columnspacing=.6, handletextpad=.2)
    title(ax, 'Measurement checks')
    letter(ax, 'd', dx=-26)

    # e · tissue state
    ax = axes_mm(fig, 72, 90, 44, 34)
    main = ratios[ratios.contrast.eq('S2-S1')].set_index('reference')
    refs = ['microdissection male (GSE150338)', 'mouse snRNA male (Census)', 'Lake healthy human cortex']
    series = [('human ÷ mouse', main.ratio_ols, INK), ('AKI ÷ control mice', aki[aki.contrast.eq('S2-S1')].set_index('reference').ratio_ols, NULL),
              ('low-injury half', halves[halves.analysis.eq('low-injury half')].set_index('reference').ratio_ols, '#7A7A7A'),
              ('high-injury half', halves[halves.analysis.eq('high-injury half')].set_index('reference').ratio_ols, LIGHT)]
    for k, (label, values, color) in enumerate(series):
        ax.bar(np.arange(len(refs)) + (k - 1.5) * .2, values.reindex(refs), .19, color=color, label=label)
    ax.set_yscale('log')
    ax.set_ylim(.05, 5)
    ax.axhline(1, color=INK, lw=.5)
    ax.set_xticks(range(len(refs)), ['mouse micro-\ndissection', 'mouse\nsnRNA', 'Lake human\ncortex'], fontsize=5.5)
    ax.set_ylabel('amplitude ratio (S2 − S1)')
    ax.legend(loc='upper left', fontsize=5.2, borderaxespad=0, ncol=2, columnspacing=.6, handlelength=1)
    title(ax, 'Tissue state does not explain the gap')
    letter(ax, 'e', dx=-26)

    # f · Lake donors
    from scipy.stats import spearmanr
    ax = axes_mm(fig, 130, 90, 46, 34)
    disease_style = {'normal': (INK, 'o'), 'acute kidney injury': (NULL, 's'), 'chronic kidney disease': ('#BDBDBD', '^')}
    for disease, (color, marker) in disease_style.items():
        block = donors[donors.disease.eq(disease)]
        ax.scatter(block.injury, block.amplitude_S2_S1, s=12, color=color, marker=marker, lw=0, label=f'{disease} ({len(block)})')
    rho = spearmanr(donors.injury, donors.amplitude_S2_S1).statistic
    ax.set(xlabel='donor injury score (within-Lake z)', ylabel='donor S2 − S1 amplitude\n(mouse axis)')
    ax.set_ylim(donors.amplitude_S2_S1.min() * .9, donors.amplitude_S2_S1.max() * 1.35)
    ax.legend(loc='upper right', fontsize=5.2, borderaxespad=0, ncol=2, columnspacing=.6, handletextpad=.2)
    title(ax, f'Lake donors: Spearman ρ = {rho:.2f}')
    letter(ax, 'f', dx=-30)

    # g · lead genes
    ax = axes_mm(fig, 14, 24, 40, 42)
    style = {'consistent with flattening': (NULL, 'o'), 'reversal': (CLASS_COLORS['reversal'], 'o'),
             'human-stronger': (ACCENT, 'o'), 'human-specific gradient': (INK, '^'), 'other beyond flattening': (INK, 's')}
    plot = lead.dropna(subset=['human S2−S1', 'mouse S2−S1'])
    untestable = plot[~plot['S2−S1 class'].isin(list(style))]
    ax.scatter(untestable['mouse S2−S1'], untestable['human S2−S1'], s=8, facecolor='white', edgecolor=LIGHT, lw=.6,
               label=f'not testable ({len(untestable)})')
    for klass, (color, marker) in style.items():
        block = plot[plot['S2−S1 class'].eq(klass)]
        if len(block):
            ax.scatter(block['mouse S2−S1'], block['human S2−S1'], s=10, color=color, marker=marker, lw=0, label=f'{klass} ({len(block)})')
    beyond = plot[plot['S2−S1 class'].isin(list(style)) & ~plot['S2−S1 class'].eq('consistent with flattening')]
    place_labels(ax, beyond['mouse S2−S1'], beyond['human S2−S1'], list(beyond.index), fontsize=4.8, step=6.5)
    xs = np.linspace(plot['mouse S2−S1'].min(), plot['mouse S2−S1'].max(), 10)
    ax.plot(xs, r * xs, color=INK, lw=.7, ls='--')
    ax.plot(xs, xs, color=NULL, lw=.6, ls=':')
    ax.axhline(0, color=LIGHT, lw=.4)
    ax.axvline(0, color=LIGHT, lw=.4)
    ax.set(xlabel='mouse S2 − S1 (log2)', ylabel='human S2 − S1 (log2)')
    ax.legend(loc='upper left', bbox_to_anchor=(0, -.2), fontsize=5, borderaxespad=0, ncol=2, columnspacing=.5, handletextpad=.1)
    title(ax, f'Lead genes (dashed: human = {r:.2f} × mouse)')
    letter(ax, 'g', dx=-26)

    # h, i · post hoc
    labels = [i.replace(' · ', '\n') for i in posthoc.index]
    yy = np.arange(len(posthoc))[::-1]
    ax = axes_mm(fig, 108, 28, 32, 36)
    ax.errorbar(posthoc.signal_sd_ratio, yy, xerr=[posthoc.signal_sd_ratio - posthoc.sd_ratio_ci_low,
                posthoc.sd_ratio_ci_high - posthoc.signal_sd_ratio], fmt='o', ms=3, color=INK, lw=.8)
    ax.set_yticks(yy, labels, fontsize=5)
    ax.axvline(1, color=NULL, lw=.5)
    ax.set(xlabel='noise-corrected SD ratio', xlim=(0, 1.1))
    title(ax, 'Spread ratio (post hoc)')
    letter(ax, 'h', dx=-120)
    ax = axes_mm(fig, 146, 28, 30, 36)
    ax.errorbar(posthoc.disattenuated_r, yy, xerr=[posthoc.disattenuated_r - posthoc.r_ci_low, posthoc.r_ci_high - posthoc.disattenuated_r],
                fmt='o', ms=3, color=INK, lw=.8)
    ax.set_yticks(yy, [''] * len(yy))
    ax.axvline(0, color=NULL, lw=.5)
    ax.set(xlabel='disattenuated correlation', xlim=(-.05, 1))
    title(ax, 'Correlation (post hoc)')
    letter(ax, 'i', dx=-8)
    save(fig, 'figure_S4')


figure_s4()

## 13 · Figure S5 · Lead-gene verification matrix (notebook 38)

One row per claimed gene and one column per pre-specified test. Dark square with ✓ = passes,
open square with × = fails, light grey = not testable, blank = not claimed for that gene. The
claimed tests follow notebook 38's rule, using its saved attenuation estimate. Verdict and probe
counts (mouse/human) are on the right.

In [ ]:
SHOWN_TESTS = [('ours', 'ours'), ('ours_attenuation', 'ours (de-attenuated)'), ('human_lake', 'Lake human'),
               ('int_vs_male_sn', 'interaction (male mouse sn)'), ('int_vs_female_sn', 'interaction (female mouse sn)'),
               ('mouse_sn_male', 'mouse sn ♂'), ('mouse_sn_female', 'mouse sn ♀'), ('mouse_micro_male', 'mouse micro ♂'),
               ('mouse_micro_female', 'mouse micro ♀'), ('mouse_absent', 'absent in mouse'), ('human_absent', 'absent in human')]
STORY_ORDER = ['creatine', 'fao', 'detox', 'reversal', 'sterol', 'lost_in_mouse', 'mouse_only_lipid', 'early_program_later',
               'oat_illustration', 'probe_example']


def claimed_tests(row, attenuation):
    """Notebook 38's display rule: the tests a claim asks for (an unclaimed test is blank, not 'not testable')."""
    shared = row.one_species == ''
    keys = ['ours']
    if row.claim_human == '0' and attenuation < 1:
        keys.append('ours_attenuation')
    if row.claim_human != '':
        keys.append('human_lake')
    if row.claim_interaction != '' and shared:
        keys += ['int_vs_male_sn', 'int_vs_female_sn']
    if row.claim_mouse != '' and row.one_species != 'human':
        keys += ['mouse_sn_male', 'mouse_sn_female', 'mouse_micro_male', 'mouse_micro_female']
    if row.one_species == 'human':
        keys.append('mouse_absent')
    if row.one_species == 'mouse':
        keys.append('human_absent')
    return keys


def figure_s5():
    F = 'Figure S5'
    evidence = table(F, 'pt_literature_deep/story_gene_evidence.csv', keep_default_na=False, dtype=str)
    manifest = json_file(F, 'pt_literature_deep/run_manifest.json')
    attenuation = float(manifest['attenuation']['a'])
    order = evidence.assign(_s=evidence.story.map({s: i for i, s in enumerate(STORY_ORDER)})).sort_values(['_s', 'role'], kind='stable')
    order = order.reset_index(drop=True)
    height = 3.05 * len(order)
    fig = new_figure(10 + height + 26)
    ax = axes_mm(fig, 40, 10, 76, height)
    for i, row in order.iterrows():
        for j, (key, _) in enumerate(SHOWN_TESTS):
            if key not in claimed_tests(row, attenuation):
                continue
            value = row[f'test_{key}']
            if value == '':
                ax.add_patch(Rectangle((j - .4, i - .4), .8, .8, color='#E3E3E3', lw=0))
            elif value == 'True':
                ax.add_patch(Rectangle((j - .4, i - .4), .8, .8, color=INK, lw=0))
                ax.text(j, i, '✓', ha='center', va='center', color='white', fontsize=5.5)
            else:
                ax.add_patch(Rectangle((j - .4, i - .4), .8, .8, facecolor='white', edgecolor=INK, lw=.6))
                ax.text(j, i, '×', ha='center', va='center', color=INK, fontsize=6)
        notes = [label for flag, label in (('female_mouse_absent', 'absent in female mouse'),
                                            ('human_sexes_disagree', 'human sexes disagree'),
                                            ('human_references_disagree', 'human references disagree')) if row[flag] == 'True']
        if row.matched_anatomy_agrees == 'False':
            notes.append('flips vs mouse PT-S2')
        text = f'{row.verdict}  ·  M{row.mouse_probes}/H{row.human_probes}' + (f'  ·  {"; ".join(notes)}' if notes else '')
        ax.text(len(SHOWN_TESTS) - .2, i, text, va='center', fontsize=5.4,
                color=INK if row.verdict in ('headline', 'supporting') else MUTED)
    ax.set_xlim(-.6, len(SHOWN_TESTS) - .4)
    ax.set_ylim(len(order) - .5, -.5)
    ax.set_yticks(range(len(order)), [f'{STORY_NAMES.get(r.story, r.story)} · {r.gene}' for r in order.itertuples()], fontsize=5.5)
    ax.set_xticks(range(len(SHOWN_TESTS)), [label for _, label in SHOWN_TESTS], rotation=55, ha='right', fontsize=5.6)
    ax.tick_params(length=0)
    for side in ax.spines.values():
        side.set_visible(False)
    for b in np.flatnonzero(order.story.to_numpy()[1:] != order.story.to_numpy()[:-1]) + .5:
        ax.axhline(b, color='#CCCCCC', lw=.5)
    ax.xaxis.tick_top()
    ax.set_xticks(range(len(SHOWN_TESTS)), [label for _, label in SHOWN_TESTS], rotation=55, ha='left', fontsize=5.6)
    verdicts = order.verdict.value_counts()
    fig.text(.005, .003, f'{int((~order.verdict.str.startswith("example")).sum())} claimed genes: '
             + ', '.join(f'{v} {k}' for k, v in verdicts.items() if not k.startswith('example'))
             + f'; attenuation a = {attenuation:.2f} (de-attenuated test shown where the human claim is flat).',
             fontsize=5.5, color=MUTED)
    save(fig, 'figure_S5')


figure_s5()

## 14 · Figure S6 · Opposite regional differences that a whole-PT comparison averages away

- **Top:** notebook 37's fitted human and mouse curves (stage cache, checked against the saved
  `T_spatial`), each specimen's means in 12 coordinate bins (section 3), and the reviewed segment
  ranges shaded.
- **Bottom:** DESeq2 log2 fold changes (human − mouse) for whole PT and for each segment (*
  padj ≤ 0.05), from notebook 31's saved table. They are checked against notebook 38's copy for
  the genes both notebooks report.

Slc22a7 is no longer shown, because its whole-PT difference is itself significant. The S3 bars
compare human cortex S3 with mouse S3 that includes outer stripe, which has no human counterpart.
A caveat on the figure says so.

In [ ]:

def figure_s6():
    F = 'Figure S6'
    genes = table(F, 'pt_pathway_final/scfates/tables/gene_level_statistics.csv', index_col=0)
    fit = cached_arrays(F, 'pt_pathway_final/scfates', 'species_fit')
    np.testing.assert_allclose(fit['T_spatial'], genes.T_spatial.to_numpy(), rtol=1e-6, atol=1e-8)
    deseq = table(F, 'pt_pathway_method_selection/conventional_deseq2_all_partitions.csv')
    evidence = table(F, 'pt_literature_deep/story_gene_evidence.csv').set_index('gene')
    deseq = deseq[deseq.partition.eq('species') & deseq.gene.isin(FIGURE_S6_GENES)]
    lfc = deseq.pivot(index='gene', columns='scope', values='log2FoldChange')
    padj = deseq.pivot(index='gene', columns='scope', values='padj')
    for gene in set(FIGURE_S6_GENES) & set(evidence.index):
        for scope in ('whole_PT', 'PT-S1', 'PT-S2', 'PT-S3'):
            np.testing.assert_allclose(lfc.loc[gene, scope], evidence.loc[gene, f'deseq2_log2fc_{scope}'], rtol=1e-6)
    bins = AGG['bins']
    record_recomputed(F, 'specimen means in 12 coordinate bins')
    spans = segment_spans(F)
    grid = fit['grid']
    index = {g: i for i, g in enumerate(genes.index)}
    fig = new_figure(84)
    for k, gene in enumerate(FIGURE_S6_GENES):
        ax = axes_mm(fig, 16 + k * 56, 44, 42, 32)
        for j, s in enumerate(('S1', 'S2', 'S3')):
            ax.axvspan(*spans[s], color='#000000', alpha=.04 + .03 * j, lw=0)
            ax.text(np.mean(spans[s]), 1.01, s, transform=ax.get_xaxis_transform(), ha='center', va='bottom', fontsize=5.5, color=MUTED)
        ax.plot(grid, fit['human'][index[gene]], color=HUMAN, lw=1.3, label='human')
        ax.plot(grid, fit['mouse'][index[gene]], color=MOUSE, lw=1.3, label='mouse')
        draw_specimen_bins(ax, bins, gene)
        ax.set(xlim=(grid[0], grid[-1]), xlabel='PT coordinate')
        ax.set_title(gene, loc='left', style='italic', pad=9)
        if k == 0:
            ax.set_ylabel('fitted log-norm')
            ax.legend(loc='upper left', fontsize=5.2, borderaxespad=0, handlelength=1)
            letter(ax, 'a', dx=-26, dy=9)
        ax = axes_mm(fig, 16 + k * 56, 12, 42, 18)
        scopes = ['whole_PT', 'PT-S1', 'PT-S2', 'PT-S3']
        values = lfc.loc[gene, scopes]
        ax.bar(range(4), values, width=.7, color=[INK] + [HUMAN if v > 0 else MOUSE for v in values[1:]])
        for j, scope in enumerate(scopes):
            mark = '*' if padj.loc[gene, scope] <= ALPHA else 'ns'
            ax.text(j, values[scope], mark, ha='center', va='bottom' if values[scope] >= 0 else 'top', fontsize=5.5)
        ax.axhline(0, color=NULL, lw=.5)
        lim = float(np.abs(values).max()) * 1.45
        ax.set_ylim(-lim, lim)
        ax.set_xticks(range(4), ['whole PT', 'S1', 'S2', 'S3'], fontsize=5.5)
        if k == 0:
            ax.set_ylabel('DESeq2 log2FC\n(human − mouse)')
            letter(ax, 'b', dx=-26)
    fig.text(.005, .006, 'S3 compares human cortex S3 with mouse S3 that includes outer stripe, which has no human counterpart; '
             'segment-specific S3 differences are therefore partly regional.', fontsize=5.5, color=MUTED)
    save(fig, 'figure_S6')


figure_s6()

## 15 · Figure S7 · Segment-label agreement with the cortex atlas (notebook 38)

Notebook 38's genome-wide check with true S1 and S3 in human cortex (Lake et al.), using reviewed
segment labels. Before drawing, the filters are rebuilt from the saved gene table and must
reproduce the saved ρ and gene counts exactly. Black points are the top 10% of genes by
`T_spatial`.

- **a** Our human S3 − S1 against Lake cortex.
- **b, c** Our species × segment interaction against Lake − male and Lake − female mouse snRNA.

In [ ]:
LAKE_CPM_FLOOR, DETECTION_FLOOR = 10., .10   # notebook 38's filters; checked by reproducing its saved table


def figure_s7():
    F = 'Figure S7'
    genes = table(F, 'pt_literature_deep/gene_level_lake_contrasts.csv', index_col=0)
    check = table(F, 'pt_literature_deep/genome_wide_lake_check.csv').set_index('check')
    top = genes.T_spatial.ge(genes.T_spatial.quantile(.9))
    lake_ok = genes.lake_max_cpm.ge(LAKE_CPM_FLOOR)
    from scipy.stats import spearmanr
    panels = [('a', 'human: ours vs Lake cortex S3 − S1', genes.ours_human_S3_minus_S1, genes.lake_human_S3_minus_S1,
               genes.detection_human.ge(DETECTION_FLOOR) & lake_ok, None, 'Lake cortex S3 − S1 (log2)', 'ours, human S3 − S1',
               'Human zonation'),
              ('b', 'interaction: ours vs Lake − male mouse snRNA', genes.ours_interaction,
               genes.lake_human_S3_minus_S1 - genes.mouse_sn_male_S3_minus_S1,
               genes[['detection_mouse', 'detection_human']].min(axis=1).ge(DETECTION_FLOOR) & lake_ok, top,
               'Lake − male mouse snRNA\n(S3 − S1, log2)', 'ours, (human − mouse)(S3 − S1)', 'Species × segment, male mice'),
              ('c', 'interaction: ours vs Lake − female mouse snRNA', genes.ours_interaction,
               genes.lake_human_S3_minus_S1 - genes.mouse_sn_female_S3_minus_S1,
               genes[['detection_mouse', 'detection_human']].min(axis=1).ge(DETECTION_FLOOR) & lake_ok, top,
               'Lake − female mouse snRNA\n(S3 − S1, log2)', 'ours, (human − mouse)(S3 − S1)', 'Species × segment, female mice')]
    fig = new_figure(66)
    for k, (panel, key, ours, external, mask, highlight, xlabel, ylabel, name) in enumerate(panels):
        keep = mask & external.notna() & ours.notna()
        if highlight is None:
            row = check.loc[key]
            assert int(row.genes) == int(keep.sum())
            np.testing.assert_allclose(spearmanr(ours[keep], external[keep]).statistic, row.Spearman, rtol=1e-9)
            text = f'ρ = {row.Spearman:.2f} ({int(row.genes):,} genes)\n{row["direction agreement"]:.1%} same sign, {int(row["strong external genes"])} strong'
        else:
            rows = [check.loc[f'{key} (all shared genes)'], check.loc[f'{key} (top 10% T_spatial)']]
            for row, subset in zip(rows, (keep, keep & highlight)):
                assert int(row.genes) == int(subset.sum())
                np.testing.assert_allclose(spearmanr(ours[subset], external[subset]).statistic, row.Spearman, rtol=1e-9)
            text = (f'all shared: ρ = {rows[0].Spearman:.2f} ({int(rows[0].genes):,})\n'
                    f'top 10%: ρ = {rows[1].Spearman:.2f} ({int(rows[1].genes):,});\n'
                    f'{rows[1]["direction agreement"]:.1%} same sign, {int(rows[1]["strong external genes"])} strong')
        ax = axes_mm(fig, 14 + k * 58, 16, 40, 40)
        if highlight is None:
            ax.scatter(external[keep], ours[keep], s=1.5, lw=0, color=HUMAN, alpha=.35, rasterized=True)
        else:
            ax.scatter(external[keep & ~highlight], ours[keep & ~highlight], s=1.5, lw=0, color=LIGHT, rasterized=True)
            ax.scatter(external[keep & highlight], ours[keep & highlight], s=2, lw=0, color=INK, rasterized=True)
        ax.axhline(0, color=NULL, lw=.4)
        ax.axvline(0, color=NULL, lw=.4)
        ax.set_xlim(np.quantile(external[keep], [.001, .999]))
        ax.set_ylim(np.quantile(ours[keep], [.001, .999]))
        ax.set(xlabel=xlabel, ylabel=ylabel)
        ax.text(.03, .97, text, transform=ax.transAxes, va='top', fontsize=5.2,
                bbox=dict(facecolor='white', edgecolor='none', alpha=.85, pad=1))
        title(ax, name)
        letter(ax, panel, dx=-26)
    save(fig, 'figure_S7')


figure_s7()

## 16 · Figure S8 · Superseded: notebook 40's zonation classes (kept for the record)

This was Figure 3 of draft v0.1. Notebook 43's symmetric analysis supersedes it: with human
thresholds matched to human amplitude and an external flat check, the mouse-only excess
disappears. A banner on the figure says so.
- **a** Per-gene S1 → S2 zonation in each species, coloured by notebook 40's S2 − S1 class. The
  species means come from section 3 and reproduce notebook 40's saved lead-gene means. The
  plotted class counts are asserted equal to `class_sizes_by_contrast.csv`.
- **b** Class sizes under notebook 40's primary rule and every alternative.
- **c** Notebook 40's representative genes: log2 CPM in each specimen at S1, S2 and cortical S3,
  from the section 3 pseudobulks.
- **d** Programs enriched in each class (top four per class; ✓ = holds with 5-bin matching and
  without low-expression genes). Dashed line: the enrichment threshold, q = 0.10.
- **e** The robust position-dependent pathways by the class of their classified members (★ core).

In [ ]:


def figure_s8():
    F = 'Figure S8'
    classes = table(F, 'pt_zonation_classes/tables/gene_zonation_classes.csv', index_col=0)
    sizes = table(F, 'pt_zonation_classes/tables/class_sizes_by_setting.csv', index_col=0)
    per_contrast = table(F, 'pt_zonation_classes/tables/class_sizes_by_contrast.csv', index_col=0)
    r3 = table(F, 'pt_zonation_classes/tables/table_R3_zonation_classes.csv', index_col=0)
    programs = table(F, 'pt_zonation_classes/tables/class_programs.csv')
    enrichment = table(F, 'pt_zonation_classes/tables/class_pathway_enrichment.csv',
                       usecols=['pathway_id', 'statistic', 'effect', 'q_joint', 'enriched'])
    assert (enrichment.enriched == (enrichment.effect.gt(0) & enrichment.q_joint.le(ENRICH_Q))).all()
    by_class = table(F, 'pt_zonation_classes/tables/robust_pathways_by_class.csv', index_col=0)
    means, segment_cpm = AGG['means'], AGG['segment_log2cpm']
    record_recomputed(F, 'species-mean gradients and specimen log2 CPM, reviewed labels')
    fig = new_figure(220)
    banner = fig.add_axes([0, 1 - 7 / 220, 1, 7 / 220])
    banner.axis('off')
    banner.add_patch(Rectangle((0, 0), 1, 1, transform=banner.transAxes, color='#F3E9D2', lw=0))
    banner.text(.01, .5, 'SUPERSEDED: notebook 40\'s asymmetric zonation classes (Figure 3 of draft v0.1).\nThe mouse-only '
                'excess is a threshold effect of lower human amplitude; see Figure 3 (notebook 43).', va='center', fontsize=6,
                fontweight='bold', color='#6B4E16', transform=banner.transAxes)

    # a · per-gene S2 − S1
    ax = axes_mm(fig, 14, 160, 62, 42)
    balanced = classes[classes.probe_balanced.eq(True) & classes['S2-S1'].notna()]
    genes = balanced.index
    x, y = means.loc[genes, 'S2-S1 mouse mean'], means.loc[genes, 'S2-S1 human mean']
    klass = balanced['S2-S1']
    assert klass.value_counts().reindex(per_contrast.columns).fillna(0).astype(int).tolist() == per_contrast.loc['S2-S1'].tolist()
    legend_counts = per_contrast.loc['S2-S1']
    subtitle = f'{len(genes):,} probe-balanced genes'
    for name in ['neither', 'indeterminate', *CLASSES]:
        mask = klass.eq(name).to_numpy()
        small = name in ('neither', 'indeterminate')
        ax.scatter(x[mask], y[mask], s=1.5 if small else 4, lw=0, color=CLASS_COLORS[name],
                   alpha=.6 if small else .9, rasterized=True, label=f'{name} ({int(legend_counts.get(name, 0)):,})')
    ax.set_xlim(np.quantile(x, [.0005, .9995]) * 1.05)
    ax.set_ylim(np.quantile(y, [.0005, .9995]) * 1.05)
    ax.axhline(0, color=NULL, lw=.5)
    ax.axvline(0, color=NULL, lw=.5)
    ax.set(xlabel='Mouse S2 − S1 (log2, mean of M1, M2)', ylabel='Human S2 − S1 (log2, mean of H1, H2)')
    ax.legend(loc='upper left', fontsize=5.5, markerscale=1.5, handletextpad=.2, borderaxespad=0, title='S2 − S1 class (genes)',
              title_fontsize=5.5)
    title(ax, f'Per-gene S1 → S2 zonation, {subtitle}')
    letter(ax, 'a', dx=-26)

    # b · class sizes by setting
    ax = axes_mm(fig, 118, 160, 46, 42)
    plot = sizes[CLASSES]
    yy = np.arange(len(plot))[::-1]
    left = np.zeros(len(plot))
    for name in CLASSES:
        ax.barh(yy, plot[name], left=left, color=CLASS_COLORS[name], height=.62, label=name, edgecolor='white', lw=.4)
        left += plot[name].to_numpy()
    for yi, (setting, row) in zip(yy, plot.iterrows()):
        ax.text(row.sum() + 15, yi, ' / '.join(str(int(v)) for v in row), va='center', fontsize=5.3)
    ax.set_yticks(yy, [s.replace('mouse confirmed by ', 'mouse conf. ') for s in plot.index], fontsize=5.6)
    ax.set_xlim(0, plot.sum(axis=1).max() * 1.45)
    ax.set_xlabel('Probe-balanced genes (S2 − S1 and S3c − early combined)')
    ax.legend(loc='lower right', fontsize=5.5, ncol=1, handlelength=1, borderaxespad=0)
    title(ax, 'Class sizes: primary rule and alternatives\n(conserved / reversal / mouse-only / human-only)')
    letter(ax, 'b', dx=-82)

    # c · representative genes
    reps = [(k, g.strip()) for k in CLASSES for g in str(r3.loc[k, 'representative genes']).split(',')[:2]]
    names = SPECIMENS['mouse'] + SPECIMENS['human']
    for k, (klass_name, gene) in enumerate(reps):
        ax = axes_mm(fig, 12 + k * 21.2, 112, 16, 26)
        for name in names:
            values = [segment_cpm.loc[gene, f'{name} {s}'] for s in ('S1', 'S2', 'S3c')]
            ax.plot([1, 2, 3], values, marker=SPECIMEN_MARKER[name], ms=2.5, lw=1,
                    color=HUMAN if SPECIES_OF[name] == 'human' else MOUSE,
                    ls='-' if name in (names[0], names[2]) else '--')
        ax.set_xticks([1, 2, 3], ['S1', 'S2', 'S3c'])
        ax.set_xlim(.7, 3.3)
        title(ax, f'{klass_name}\n{gene}')
        if k == 0:
            ax.set_ylabel('log2 CPM')
            letter(ax, 'c', dx=-26, dy=12)
    fig.text(12 / WIDTH_MM, 149 / fig.height_mm, 'Representative genes in each specimen: orange human sections, blue mice; '
             'solid M1, H1; dashed M2, H2; S3c = cortical-like mouse S3', fontsize=6, fontweight='bold')

    # d · programs per class
    ax = axes_mm(fig, 52, 16, 36, 84)
    top = programs.groupby('class', sort=False).head(4)
    top = top.assign(order=top['class'].map({k: i for i, k in enumerate(CLASSES)})).sort_values(['order', 'best_q'], kind='stable')
    yy = np.arange(len(top))[::-1]
    ax.barh(yy, -np.log10(top.best_q.clip(lower=1e-12)), color=[CLASS_COLORS[c] for c in top['class']], height=.62)
    ax.set_yticks(yy, [(n[:40] + ('…' if len(n) > 40 else '')) + (' ✓' if r else '') for n, r in zip(top.name, top.robust_to_expression)],
                  fontsize=5.5)
    ax.axvline(-np.log10(ENRICH_Q), color=MUTED, ls='--', lw=.6)
    ax.set_xlabel('−log10 q (joint matched test)\n✓ also with 5-bin matching and without low-expression genes')
    handles = [Patch(color=CLASS_COLORS[k], label=k) for k in CLASSES]
    ax.legend(handles=handles, loc='lower right', fontsize=5.5, handlelength=1, borderaxespad=0)
    title(ax, 'Programs enriched in each class')
    letter(ax, 'd', dx=-140)

    # e · robust pathways by class
    ax = axes_mm(fig, 140, 16, 37, 84)
    share_columns = [f'share_{k}' for k in CLASS_COLORS]
    rm = by_class.copy()
    unclassifiable = rm.classified_members.fillna(0).eq(0)
    rm[share_columns] = rm[share_columns].fillna(0)
    rm = rm.assign(_unc=unclassifiable).sort_values(['_unc', 'share_mouse-only', 'share_conserved'], ascending=[False, True, True])
    yy = np.arange(len(rm))
    left = np.zeros(len(rm))
    for column in share_columns:
        name = column.replace('share_', '')
        ax.barh(yy, rm[column], left=left, color=CLASS_COLORS[name], height=.78, label=name)
        left += rm[column].to_numpy()
    for yi, (pid, row) in zip(yy, rm.iterrows()):
        if row._unc:
            ax.text(.02, yi, 'no member classifiable in both species', va='center', fontsize=5, color=MUTED)
    labels = [('★ ' if c else '') + (n[:38] + ('…' if len(n) > 38 else '')) for n, c in zip(rm.name, rm.core)]
    ax.set_yticks(yy, labels, fontsize=5.2)
    ax.set(xlim=(0, 1), ylim=(-.6, len(rm) - .4), xlabel='Share of classified members')
    ax.legend(loc='upper left', bbox_to_anchor=(-.35, -.09), ncol=3, fontsize=5.2, handlelength=.9, columnspacing=.6,
              borderaxespad=0)
    title(ax, f'{len(rm)} robust pathways by class')
    letter(ax, 'e', dx=-138)
    save(fig, 'figure_S8')


figure_s8()

## 17 · Manifest, missing inputs and layout report

`figure_manifest.json` lists each figure's source files. Files marked as stage cache are arrays
that an upstream notebook keeps only in its cache; they should be saved as tables. The file also
records the layout check and the key counts. `missing_inputs.csv` lists the quantities that no
upstream notebook saves, with a proposed file name for each.

In [ ]:
missing = pd.DataFrame(MISSING).drop_duplicates()
missing.to_csv(output / 'missing_inputs.csv', index=False)
manifest = {'notebook_logic_version': NOTEBOOK_LOGIC_VERSION, 'figures': OUTPUTS, 'sources': SOURCES,
            'missing_inputs': missing.to_dict(orient='records'), 'layout': LAYOUT, 'key_counts': KEY_COUNTS,
            'figure_4_diagnostics': FIGURE_4_DIAGNOSTICS,
            'caveat': 'Two control mice and two cortex sections from one male human donor; descriptive.'}
(output / 'figure_manifest.json').write_text(json.dumps(manifest, indent=2, default=str))
print('Figures:', ', '.join(OUTPUTS))
print(f'{len(missing)} missing inputs:')
for row in missing.itertuples():
    print(f'  {row.figure} {row.panel}: {row.quantity} → {row.proposed_file} ({row.upstream})')
stage_cache_sources = sorted({(f, s['file']) for f, items in SOURCES.items() for s in items if s['kind'].startswith('stage cache')})
print('Stage-cache sources (should become saved tables):')
for figure, path in stage_cache_sources:
    print(f'  {figure}: {path}')